<a href="https://colab.research.google.com/github/ts387/cryosphere/blob/main/cryoSPHERE_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# cryoSPHERE on Google Colab

**cryoSPHERE** reconstructs the conformational heterogeneity of a protein from single-particle cryo-EM images. Starting from one atomic model, it learns how segments of the protein move (rigid rotations and translations per segment) to explain every particle image, and gives you a latent space of conformations plus structures you can open in ChimeraX or PyMOL.

![cryoSPHERE overview](https://github.com/gabriel-ducrocq/cryoSPHERE/blob/main/cryosphere/figures/VAEFlow3.jpg?raw=true)

This notebook installs cryoSPHERE, prepares your data, trains a model on a Colab GPU and lets you explore the results, without leaving the browser.

### How to use it
1. **Runtime → Change runtime type → T4 GPU** (or better), then work through the steps in order. Each step is a form: set the fields, then press ▶.
2. To try everything on the published example dataset, keep the defaults and use **Runtime → Run all**. Most of the time goes into training (5.1), whose status line shows the time per epoch and an ETA once the first epoch is done. For a quick end-to-end check first, set `N_epochs = 2` and `max_particles = 2000` in 4.1.
3. For your own data, put your files in a Google Drive folder and pick **Google Drive** in 1.4 and 2.1.

| Step | What happens |
|---|---|
| **1 · Setup** | GPU check, helpers, installation, project folder, optional Weights & Biases |
| **2 · Data** | download/locate files, validate and auto-fix metadata, preview particles |
| **3 · Structure** | center the atomic model, check its fit, choose a low-pass filter from the FSC |
| **4 · Configure** | segmentation, training and image settings → `parameters.yaml` + `images.yaml` |
| **5 · Train** | live progress, loss curves, automatic checkpoint clean-up, background mode |
| **6 · Analyze** | latent space PCA, 3D motion movies, segmentation, per-particle and cluster structures, maps |
| **7 · Export** | download or save results to Drive |
| **8 · Troubleshooting** | common errors and fixes |

### What you need for your own data
* **Particle images**: `.mrcs` stack(s).
* **Poses and CTF** from a consensus refinement: a RELION `.star` or a cryoSPARC `.cs` file.
* **An atomic model** (`.pdb`, or `.cif` which is converted for you) fitted into the consensus map, e.g. with ChimeraX.
* *Optional*: the **consensus map** (`.mrc`), used to center the model and to check the fit.

cryoSPHERE: [paper (ICLR 2025)](https://arxiv.org/abs/2407.01574) · [code](https://github.com/gabriel-ducrocq/cryoSPHERE) · [example dataset](https://zenodo.org/records/15794610)

---
## 1 · Set up the runtime

In [ ]:
#@title 1.1 · Check that a GPU is attached { display-mode: "form" }
#@markdown cryoSPHERE trains on CUDA GPUs only. If this cell fails, choose **Runtime → Change runtime type → T4 GPU (or better) → Save** and run it again.
import os, sys, shutil, subprocess

def _main():
    def sh(cmd):
        return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
    print(f'Python      {sys.version.split()[0]}')
    try:
        import torch
        print(f'PyTorch     {torch.__version__} (CUDA {torch.version.cuda})')
        cuda = torch.cuda.is_available()
    except ImportError:
        cuda = False
    mem = sh("awk '/MemTotal/ {printf \"%.1f GB\", $2/1048576}' /proc/meminfo")
    disk = shutil.disk_usage('/content' if os.path.exists('/content') else '/').free / 1e9
    print(f'CPU cores   {os.cpu_count()} · RAM {mem} · free disk {disk:.0f} GB')
    gpus = sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
    if not gpus or not cuda:
        raise RuntimeError('No CUDA GPU is available. Choose Runtime → Change runtime type → T4 GPU, then re-run from 1.1.')
    for i, g in enumerate(gpus.splitlines()):
        print(f'GPU {i}       {g}')
    if len(gpus.splitlines()) > 1:
        print('cryoSPHERE will train on all GPUs listed above.')
    print('\n✅ GPU runtime ready.')

_main()

In [ ]:
#@title 1.2 · Load the notebook helpers { display-mode: "form" }
#@markdown Defines the helper functions the rest of the notebook uses (data detection, validation, FSC, training monitor, viewers).
#@markdown **Re-run this cell after every runtime restart**, then re-run 1.4 to reload your project.
import os, sys, re, io, json, glob, time, math, shlex, shutil, signal, socket, select, hashlib, tarfile, zipfile, subprocess, warnings, importlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import yaml
from IPython.display import display, HTML, Markdown, Image as IPyImage

importlib.invalidate_caches()
IN_COLAB = 'google.colab' in sys.modules
STATE = globals().get('STATE') or {}
STATE_FILE_NAME = 'cryosphere_colab_state.json'
CRYOSPHERE_PORT = 12355            # cryoSPHERE hard-codes this rendezvous port for torch.distributed
SKIP_DIRS = {'cryoSPHERE', 'analysis', 'runs', '__MACOSX', '.ipynb_checkpoints', 'cryosphere_inputs', 'fsc_check', 'centered'}
CS_TOOLS = ('cryosphere_train', 'cryosphere_analyze', 'cryosphere_center_origin', 'cryosphere_structure_to_volume')
DRIVE_ROOT = '/content/drive'
LOCAL_ROOT = Path('/content') if os.path.exists('/content') else Path.cwd()
WORK_ROOT = LOCAL_ROOT / 'cryosphere_work'          # local mirror of runs whose project lives on Google Drive
BG_REGISTRY = Path('/tmp/cryosphere_colab_background.json')   # per runtime, independent of the open project


# ----------------------------------------------------------------------------- messages & state
class StepError(RuntimeError):
    """A problem with the inputs or settings of a step (not a bug); the message says how to fix it."""


def ok(msg): print(f'✅ {msg}')
def warn(msg): print(f'⚠️  {msg}')
def info(msg): print(f'ℹ️  {msg}')
def fail(msg):
    raise StepError(msg)


def run_step(main):
    """Run a cell's _main(); a StepError is shown as a highlighted message (then re-raised so 'Run all' stops)."""
    try:
        main()
    except StepError as e:
        import html
        display(HTML(f'<div style="border-left:4px solid #d93025;padding:6px 10px;margin:6px 0;color:#d93025;'
                     f'font-weight:600;white-space:pre-wrap">❌ {html.escape(str(e))}</div>'))
        raise


_STEP_FOR = {'project_dir': '1.4', 'data_dir': '2.1', 'particles_file': '2.2', 'metadata': '2.2', 'structure_file': '2.2',
             'dataset': '2.2', 'structure_centered': '3.1', 'run_dir': '4.1', 'analysis_dir': '6.1'}
_PATH_KEYS = {'data_dir', 'particles_file', 'metadata', 'structure_file', 'structure_centered'}


def need(*keys):
    for k in keys:
        if not STATE.get(k):
            fail(f"Missing '{k}': run step {_STEP_FOR.get(k, 'above')} first.")
        if k in _PATH_KEYS and not Path(STATE[k]).exists():
            fail(f'{STATE[k]} no longer exists (a new Colab runtime starts with an empty disk). '
                 f'Re-run step {_STEP_FOR[k]} (and the steps after it) with the same settings.')


def save_state():
    if STATE.get('project_dir'):
        p = Path(STATE['project_dir']) / STATE_FILE_NAME
        p.write_text(json.dumps(STATE, indent=2, default=str))


def load_state(project_dir):
    p = Path(project_dir) / STATE_FILE_NAME
    if p.exists():
        STATE.clear()
        STATE.update(json.loads(p.read_text()))
        return True
    return False


def human_size(n):
    for unit in ('B', 'KB', 'MB', 'GB', 'TB'):
        if abs(n) < 1024 or unit == 'TB':
            return f'{n:.1f} {unit}' if unit != 'B' else f'{int(n)} B'
        n /= 1024


def show_table(rows, columns=None):
    df = pd.DataFrame(rows, columns=columns)
    display(df.style.hide(axis='index') if hasattr(df.style, 'hide') else df)


def disk_free(path='/content' if os.path.exists('/content') else '/'):
    return shutil.disk_usage(path).free


# ----------------------------------------------------------------------------- shell
def run_cmd(cmd, cwd=None, env=None, check=True, echo=True):
    """Run a short command and stream its output. Returns (returncode, output)."""
    printable = cmd if isinstance(cmd, str) else ' '.join(shlex.quote(str(c)) for c in cmd)
    if echo:
        print(f'$ {printable}')
    proc = subprocess.Popen(cmd, shell=isinstance(cmd, str), cwd=cwd, env={**os.environ, 'PYTHONWARNINGS': 'ignore', **(env or {})},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, text=True, bufsize=1)
    out = []
    for line in proc.stdout:
        out.append(line)
        if echo:
            print(line, end='')
    proc.wait()
    if check and proc.returncode != 0:
        tail = ''.join(out[-15:]) if not echo else ''
        fail(f'Command failed (exit code {proc.returncode}): {printable}\n{tail}')
    return proc.returncode, ''.join(out)


def port_in_use(port=CRYOSPHERE_PORT):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            s.bind(('127.0.0.1', port))
            return False
        except OSError:
            return True


def _tool_of(cmdline):
    """The cryoSPHERE command a process runs (matched on the executable, not on paths in its arguments)."""
    for a in (cmdline or [])[:2]:
        if os.path.basename(a) in CS_TOOLS:
            return os.path.basename(a)
    return None


def cryosphere_processes():
    import psutil
    procs = []
    for p in psutil.process_iter(['pid', 'cmdline', 'status']):
        if p.info['pid'] != os.getpid() and p.info['status'] != psutil.STATUS_ZOMBIE and _tool_of(p.info['cmdline']):
            procs.append(p)
    return procs


def describe_process(p):
    cmd = p.info.get('cmdline') or []
    yml = next((cmd[i + 1] for i, a in enumerate(cmd[:-1]) if a == '--experiment_yaml'), '')
    return f'pid {p.pid} ({_tool_of(cmd)}{", " + yml if yml else ""})'


def bg_record():
    try:
        return json.loads(BG_REGISTRY.read_text())
    except Exception:
        return None


def set_bg_record(rec):
    if rec is None:
        BG_REGISTRY.unlink(missing_ok=True)
    else:
        BG_REGISTRY.write_text(json.dumps(rec))


def background_run_alive():
    """True while the background training started by 5.1 in this runtime (in any project) is running."""
    rec = bg_record()
    if not rec:
        return False
    popen = globals().get('_BG_POPEN')
    if popen is not None and popen.pid == rec['pid']:
        return popen.poll() is None
    try:
        import psutil
        p = psutil.Process(rec['pid'])
        return p.status() != psutil.STATUS_ZOMBIE and _tool_of(p.cmdline()) == 'cryosphere_train'
    except Exception:
        return False


def _port_holder(port=CRYOSPHERE_PORT):
    import psutil
    try:
        for c in psutil.net_connections('tcp'):
            if c.laddr and c.laddr.port == port and c.status == psutil.CONN_LISTEN and c.pid:
                return psutil.Process(c.pid)
    except Exception:
        pass
    return None


def ensure_gpu_free_for_cryosphere(action):
    """cryoSPHERE always rendezvous on localhost:12355, so only one cryoSPHERE job can run at a time."""
    if background_run_alive():
        rec = bg_record()
        fail(f'A background training run is still active ({Path(rec["run_dir"]).name}, pid {rec["pid"]}). cryoSPHERE uses '
             f'a fixed network port ({CRYOSPHERE_PORT}), so {action} cannot run at the same time. Wait for it to finish '
             '(follow it in 5.2) or stop it in 5.3.')
    procs = cryosphere_processes()
    trains = [p for p in procs if _tool_of(p.info['cmdline']) == 'cryosphere_train']
    if trains:
        fail(f'A cryosphere_train process is running that this notebook did not start in the background: '
             f'{"; ".join(describe_process(p) for p in trains)}. Wait for it, or stop it in 5.3 (tick confirm_stop).')
    if procs:
        warn(f'Stopping {len(procs)} leftover cryoSPHERE process(es) from an interrupted cell: '
             + '; '.join(describe_process(p) for p in procs))
        for p in procs:
            kill_process_group(p.pid, grace=2)
    for _ in range(30):
        if not port_in_use():
            return
        time.sleep(1)
    holder = _port_holder()
    who = f' It is held by pid {holder.pid} ({" ".join(holder.cmdline())[:120]}).' if holder else ''
    fail(f'Port {CRYOSPHERE_PORT} is still in use.{who} Use Runtime → Disconnect and delete runtime, then start again '
         'from 1.1 (your project folder on Drive is kept).')


def kill_process_group(pid, grace=10):
    try:
        os.killpg(os.getpgid(pid), signal.SIGINT)
    except Exception:
        return
    for _ in range(grace * 2):
        try:
            os.kill(pid, 0)
        except OSError:
            break
        time.sleep(0.5)
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(os.getpgid(pid), sig)
            time.sleep(1)
        except Exception:
            break


# ----------------------------------------------------------------------------- downloads & archives
def download(url, dest, expected_size=None, md5=None, desc=None):
    import requests
    from tqdm.auto import tqdm
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and expected_size and dest.stat().st_size == expected_size:
        info(f'{dest.name}: already downloaded, skipping.')
        return dest
    tmp = dest.with_name(dest.name + '.part')
    h = hashlib.md5() if md5 else None
    try:
        with requests.get(url, stream=True, timeout=60, allow_redirects=True) as r:
            r.raise_for_status()
            total = int(r.headers.get('content-length') or expected_size or 0)
            with open(tmp, 'wb') as f, tqdm(total=total or None, unit='B', unit_scale=True, unit_divisor=1024,
                                            desc=desc or dest.name, leave=True) as bar:
                for chunk in r.iter_content(chunk_size=4 << 20):
                    f.write(chunk)
                    if h:
                        h.update(chunk)
                    bar.update(len(chunk))
    except BaseException as e:
        tmp.unlink(missing_ok=True)
        if isinstance(e, (KeyboardInterrupt, StepError)):
            raise
        fail(f'Download of {dest.name} failed: {e}')
    if expected_size and tmp.stat().st_size != expected_size:
        tmp.unlink(missing_ok=True)
        fail(f'{dest.name}: the download stopped early ({human_size(tmp.stat().st_size if tmp.exists() else 0)} of '
             f'{human_size(expected_size)}). Re-run the cell to try again.')
    if h and h.hexdigest() != md5:
        tmp.unlink(missing_ok=True)
        fail(f'Checksum mismatch for {dest.name}; the download was corrupted. Re-run the cell.')
    tmp.replace(dest)
    return dest


def remote_size(url):
    """Size announced by the server (0 if unknown)."""
    import requests
    try:
        r = requests.head(url, allow_redirects=True, timeout=30)
        return int(r.headers.get('content-length') or 0)
    except Exception:
        return 0


def is_archive(path):
    n = str(path).lower()
    return n.endswith(('.zip', '.tar', '.tar.gz', '.tgz', '.tar.bz2', '.tbz2', '.tar.xz', '.txz'))


def extract(path, dest=None):
    path = Path(path)
    dest = Path(dest or path.parent)
    marker = dest / f'.{path.name}.extracted.done'
    if marker.exists():
        return
    info(f'Extracting {path.name} …')
    if str(path).lower().endswith('.zip'):
        with zipfile.ZipFile(path) as z:
            z.extractall(dest)
    else:
        skipped = []
        def safe(member, target):
            try:
                return tarfile.data_filter(member, target)
            except tarfile.FilterError:           # e.g. RELION 'Movies -> /data/raw' links: skip, keep going
                skipped.append(member.name)
                return None
        with tarfile.open(path) as t:
            if hasattr(tarfile, 'data_filter'):
                t.extractall(dest, filter=safe)
            else:
                t.extractall(dest)
        if skipped:
            info(f'Skipped {len(skipped)} link(s) pointing outside the archive (e.g. {skipped[0]}).')
    marker.touch()


def zenodo_files(record_id):
    """(title, [{name, size, url, md5}]) for a Zenodo record; handles both the legacy list and the newer 'entries' layout."""
    import requests
    from urllib.parse import quote
    r = requests.get(f'https://zenodo.org/api/records/{record_id}', timeout=60)
    if r.status_code != 200:
        fail(f'Zenodo record {record_id} not found (HTTP {r.status_code}).')
    rec = r.json()

    def entries_of(field):
        if isinstance(field, dict):
            field = field.get('entries', [])
        return list(field.values()) if isinstance(field, dict) else list(field or [])

    entries = entries_of(rec.get('files'))
    if not entries:
        r2 = requests.get(f'https://zenodo.org/api/records/{record_id}/files', timeout=60)
        if r2.status_code == 200:
            entries = entries_of(r2.json())
    files = []
    for f in entries:
        key = f.get('key') or f.get('filename')
        links = f.get('links', {})
        url = links.get('content') or (links.get('self') if str(links.get('self', '')).endswith('/content') else None) \
            or f'https://zenodo.org/records/{record_id}/files/{quote(key)}?download=1'
        checksum = f.get('checksum') or ''
        files.append({'name': key, 'size': int(f.get('size') or f.get('filesize') or 0), 'url': url,
                      'md5': checksum.split(':', 1)[1] if checksum.startswith('md5:') else None})
    if not files:
        fail(f'Zenodo record {record_id} lists no downloadable files (it may be restricted).')
    return rec.get('metadata', {}).get('title', f'record {record_id}'), files


# ----------------------------------------------------------------------------- file discovery
def find_files(root, suffixes):
    root = Path(root)
    hits = []
    for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS and not d.startswith('.')]
        for fn in filenames:
            if fn.lower().endswith(suffixes) and not fn.startswith('._'):
                hits.append(Path(dirpath) / fn)
    return sorted(hits)


def rank_by_name(paths, prefer=(), avoid=()):
    def score(p):
        n = p.name.lower()
        return (-sum(k in n for k in prefer) + sum(k in n for k in avoid), len(p.parts), n)
    return sorted(paths, key=score)


def lenient_yaml_load(path):
    """Load cryoSPHERE example YAMLs, which mix tabs and spaces and can contain wrapped comment lines."""
    text = Path(path).read_text()
    try:
        return yaml.safe_load(text) or {}
    except yaml.YAMLError:
        pass
    blocks, cur = [], []
    for line in text.expandtabs(8).splitlines():
        if re.match(r'^[A-Za-z_][\w\-]*\s*:', line):
            if cur:
                blocks.append(cur)
            cur = [line]
        elif cur:
            cur.append(line)
    if cur:
        blocks.append(cur)
    out = {}
    for b in blocks:
        for candidate in (b, [l for l in b if ':' in l or not l.strip() or l.strip().startswith(('#', '-'))]):
            try:
                out.update(yaml.safe_load('\n'.join(candidate)) or {})
                break
            except yaml.YAMLError:
                continue
    return out


def write_yaml(path, obj):
    Path(path).write_text(yaml.safe_dump(obj, sort_keys=False, default_flow_style=None))


# ----------------------------------------------------------------------------- particle metadata
STAR_REQUIRED = ['rlnImageName', 'rlnAngleRot', 'rlnAngleTilt', 'rlnAnglePsi', 'rlnDefocusU', 'rlnDefocusV', 'rlnDefocusAngle']
STAR_OPTICS = ['rlnVoltage', 'rlnSphericalAberration', 'rlnAmplitudeContrast']
CS_CTF = ['ctf/df1_A', 'ctf/df2_A', 'ctf/df_angle_rad', 'ctf/accel_kv', 'ctf/cs_mm', 'ctf/amp_contrast', 'ctf/phase_shift_rad']
CS_BLOB = ['blob/path', 'blob/idx']


def read_star(path):
    import starfile
    data = starfile.read(path, always_dict=True)
    if 'particles' in data:
        particles = data['particles']
    elif len(data) == 1:
        particles = next(iter(data.values()))
    else:
        fail(f'{Path(path).name} has no data_particles block (blocks: {list(data)}).')
    return data.get('optics'), particles


def read_cs(path):
    arr = np.load(path)
    if arr.dtype.names is None:
        fail(f'{Path(path).name} is not a cryoSPARC .cs file (no named fields).')
    return arr


def cs_alignment_key(arr):
    names = arr.dtype.names
    if 'alignments3D/pose' in names:
        return 'alignments3D'
    if 'alignments_class_0/pose' in names:
        return 'alignments_class_0'
    return None


def merge_cs_passthrough(main, passthrough):
    """Add the fields of a cryoSPARC passthrough file that are missing from the main file (joined on uid)."""
    if 'uid' not in main.dtype.names or 'uid' not in passthrough.dtype.names:
        fail('Both .cs files need a uid field to be merged.')
    order = np.argsort(passthrough['uid'])
    pos = np.searchsorted(passthrough['uid'][order], main['uid'])
    pos = np.clip(pos, 0, len(order) - 1)
    rows = order[pos]
    if not np.all(passthrough['uid'][rows] == main['uid']):
        fail('Some particles of the main .cs file are missing from the passthrough file (uid mismatch).')
    extra = [n for n in passthrough.dtype.names if n not in main.dtype.names]
    descr = [d for d in main.dtype.descr if d[0]] + [d for d in passthrough.dtype.descr if d[0] in extra]
    merged = np.zeros(len(main), dtype=np.dtype(descr))
    for n in main.dtype.names:
        merged[n] = main[n]
    for n in extra:
        merged[n] = passthrough[n][rows]
    return merged


def save_cs(arr, path):
    with open(path, 'wb') as f:      # np.save on a file handle keeps the .cs extension
        np.save(f, arr)


def image_refs(kind, particles):
    """List of (frame_index, relative_path) for every particle, as cryoSPHERE reads them."""
    if kind == 'star':
        refs = []
        for name in particles['rlnImageName']:
            idx, path = str(name).split('@', 1)
            refs.append((int(idx) - 1, path))
        return refs
    paths = [p.decode('ascii') if isinstance(p, bytes) else str(p) for p in particles['blob/path']]
    return [(int(i), p.replace('>', '')) for i, p in zip(particles['blob/idx'], paths)]


def resolve_particles_folder(refs, candidates):
    """Pick the folder that cryoSPHERE should prepend to the image paths in the metadata file."""
    sample = sorted({p for _, p in refs})
    for c in candidates:
        if c and all(os.path.exists(os.path.join(c, p)) for p in sample[:50]):
            return str(c)
    return None


def relocate_stacks(refs, search_root):
    """Map referenced stack paths to files under search_root, matching the longest trailing part of the path.
    Returns (mapping, problem): mapping is None when a stack is missing or ambiguous."""
    found = {}
    for f in find_files(search_root, ('.mrcs', '.mrc', '.st')):
        found.setdefault(f.name, []).append(f)
    mapping = {}
    for p in sorted({p for _, p in refs}):
        cands = found.get(os.path.basename(p), [])
        if not cands:
            return None, f'{p} is not in the data folder'
        want = Path(p).parts
        def depth(f):
            got, k = f.parts, 0
            while k < min(len(got), len(want)) and got[-1 - k] == want[-1 - k]:
                k += 1
            return k
        scores = sorted(((depth(f), f) for f in cands), key=lambda t: -t[0])
        if len(scores) > 1 and scores[0][0] == scores[1][0]:
            return None, (f'{p} matches several files ({scores[0][1]}, {scores[1][1]}, …); set particles_folder to the '
                          'folder the paths in the metadata are relative to')
        mapping[p] = str(scores[0][1])
    return mapping, None


def missing_stacks(refs, folder):
    return [p for p in sorted({p for _, p in refs}) if not os.path.exists(os.path.join(folder, p))]


def read_particle_image(path, frame):
    import mrcfile
    with mrcfile.mmap(path, mode='r', permissive=True) as m:
        data = m.data
        img = np.array(data[frame] if data.ndim > 2 else data, dtype=np.float32)
    return img[0] if img.ndim == 3 else img


def stack_header(path):
    import mrcfile
    with mrcfile.mmap(path, mode='r', permissive=True) as m:
        return tuple(int(s) for s in m.data.shape), float(m.voxel_size.x)


# ----------------------------------------------------------------------------- structures
def load_polymer(pdb_path):
    """Read the structure exactly the way cryoSPHERE does (C-alpha / C1' atoms only)."""
    from cryosphere.model.polymer import Polymer
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return Polymer.from_pdb(str(pdb_path))


def chain_table(poly):
    rows = []
    for ch in pd.unique(poly.chain_id):
        sel = poly.chain_id == ch
        rows.append({'chain': ch, 'residues used by cryoSPHERE': int(sel.sum()),
                     'first residue #': int(poly.res_id[sel].min()), 'last residue #': int(poly.res_id[sel].max())})
    return rows


def shift_pdb(src, dst, shift):
    """Translate every atom of a PDB file (all models are kept)."""
    from biotite.structure.io.pdb import PDBFile
    f = PDBFile.read(str(src))
    arr = f.get_structure()
    arr.coord += np.asarray(shift, dtype=np.float32)
    out = PDBFile()
    out.set_structure(arr)
    out.write(str(dst))


def renumber_pdb(src, dst):
    """Make residue numbers unique within each chain: numbers are shifted up after insertion codes (52A, 52B, …) or
    repeats, while gaps (missing loops) are kept, because cryoSPHERE links residues whose numbers differ by 1."""
    import biotite.structure as bs
    from biotite.structure.io.pdb import PDBFile
    arr = PDBFile.read(str(src)).get_structure(model=1)
    starts = bs.get_residue_starts(arr, add_exclusive_stop=True)
    new_ids, shift, last = np.empty(len(arr), dtype=int), {}, {}
    for a, b in zip(starts[:-1], starts[1:]):
        ch = arr.chain_id[a]
        if arr.ins_code[a] or (ch in last and arr.res_id[a] + shift.get(ch, 0) <= last[ch]):
            shift[ch] = last[ch] + 1 - arr.res_id[a]
        new_ids[a:b] = last[ch] = arr.res_id[a] + shift.get(ch, 0)
    arr.res_id = new_ids
    arr.ins_code[:] = ''
    out = PDBFile()
    out.set_structure(arr)
    out.write(str(dst))


def to_original_frame(src, dst, shift):
    """Copy a cryoSPHERE-frame PDB or map back into the frame of the original consensus map (subtract the centering shift)."""
    src, dst = Path(src), Path(dst)
    if src.suffix == '.pdb':
        shift_pdb(src, dst, -np.asarray(shift, dtype=np.float32))
    else:
        import mrcfile
        with mrcfile.open(src, permissive=True) as m:
            data, vox = m.data.copy(), m.voxel_size.copy()
            org = np.array([float(m.header.origin.x), float(m.header.origin.y), float(m.header.origin.z)])
        with mrcfile.new(dst, overwrite=True) as m:
            m.set_data(data)
            m.voxel_size = vox
            m.header.origin = tuple(org - np.asarray(shift, dtype=float))


def ca_pdb_text(chain_ids, res_ids, res_names, coords, bfactors=None, model=None):
    lines = [f'MODEL     {model:4d}'] if model is not None else []
    for i, (c, r, n, xyz) in enumerate(zip(chain_ids, res_ids, res_names, coords)):
        b = 0.0 if bfactors is None else float(bfactors[i])
        lines.append('ATOM  %5d  CA  %3s %1s%4d    %8.3f%8.3f%8.3f  1.00%6.2f           C' %
                     ((i + 1) % 100000, str(n)[:3], str(c)[:1], int(r) % 10000, *xyz, b))
    lines.append('ENDMDL' if model is not None else 'END')
    return '\n'.join(lines)


def read_ca_coords(pdb_path):
    return load_polymer(pdb_path).coord.astype(np.float32)


# ----------------------------------------------------------------------------- segmentation helpers
def parse_parts(spec, poly, numbering):
    """'A:1-120:4, B:all:2' → cryoSPHERE segmentation_config (start/end are 0-based positions within the chain)."""
    parts, used = {}, {}
    for k, item in enumerate([s.strip() for s in re.split(r'[;,\n]+', spec) if s.strip()], start=1):
        m = re.fullmatch(r'([^:\s]+)\s*:\s*(all|\*|-?\d+\s*-\s*-?\d+)\s*:\s*(\d+)', item, flags=re.I)
        if not m:
            fail(f"Could not read segmentation part '{item}'. Use CHAIN:START-END:N_SEGMENTS, e.g. A:1-250:4 or A:all:4.")
        chain, rng, n = m.group(1), m.group(2), int(m.group(3))
        sel = np.where(poly.chain_id == chain)[0]
        if len(sel) == 0:
            fail(f"Chain '{chain}' is not in the structure (chains: {', '.join(pd.unique(poly.chain_id))}).")
        res = poly.res_id[sel]
        if rng.lower() in ('all', '*'):
            s, e = 0, len(sel) - 1
        else:
            a, b = (int(x) for x in re.fullmatch(r'(-?\d+)\s*-\s*(-?\d+)', rng.strip()).groups())
            if numbering.startswith('PDB'):
                inside = np.where((res >= a) & (res <= b))[0]
                if len(inside) == 0:
                    fail(f"No residues numbered {a}–{b} in chain {chain} (its residues run {res.min()}–{res.max()}).")
                s, e = int(inside[0]), int(inside[-1])
            else:
                s, e = a, b
                if not (0 <= s <= e < len(sel)):
                    fail(f'Positions {s}–{e} are outside chain {chain} (valid: 0–{len(sel) - 1}).')
        if n < 1 or n > e - s + 1:
            fail(f'Part {item}: the number of segments must be between 1 and the number of residues ({e - s + 1}).')
        for (s2, e2) in used.get(chain, []):
            if s <= e2 and s2 <= e:
                fail(f'Part {item} overlaps another part on chain {chain}.')
        used.setdefault(chain, []).append((s, e))
        parts[f'part{k}'] = {'chain': str(chain), 'start_res': int(s), 'end_res': int(e), 'N_segm': n}
    if not parts:
        fail('No segmentation parts given.')
    return parts


def normalize_segmentation(seg_cfg, poly):
    """Validate a segmentation_config read from a YAML file and drop settings cryoSPHERE 0.6.x cannot run."""
    if not isinstance(seg_cfg, dict) or not seg_cfg:
        fail('The YAML file has no usable segmentation_config.')
    out, notes = {}, []
    for name, part in seg_cfg.items():
        part = dict(part)
        n = int(part['N_segm'])
        if part.get('all_protein'):
            if len(seg_cfg) > 1:
                fail('all_protein: True cannot be combined with other parts.')
        else:
            chain = part['chain'] = str(part['chain'])
            n_chain = int((poly.chain_id == chain).sum())
            if n_chain == 0:
                fail(f"{name}: chain '{chain}' is not in the structure.")
            if not (0 <= int(part['start_res']) <= int(part['end_res']) < n_chain):
                fail(f"{name}: start_res/end_res {part['start_res']}–{part['end_res']} are outside chain {chain} "
                     f'(0–{n_chain - 1}, counted from 0 within the chain).')
        sv = part.get('segmentation_start_values')
        if sv is not None:
            keys = ['means_means', 'means_stds', 'stds_means', 'stds_stds', 'proportions_means', 'proportions_stds']
            if not all(k in sv and len(sv[k]) == n for k in keys):
                notes.append(f'{name}: custom start values ignored (each list needs {n} entries).')
                part.pop('segmentation_start_values')
            elif min((float(m) + 17.4) / max(abs(float(d)), 1e-6) for m, d in zip(sv['stds_means'], sv['stds_stds'])) < 7:
                notes.append(f'{name}: custom start values ignored: their stds_stds are so wide that sampled segment widths '
                             'underflow to 0 in float32, which makes cryoSPHERE produce NaN.')
                part.pop('segmentation_start_values')
        prior = part.get('segmentation_prior')
        if prior is not None and (not isinstance(prior, dict) or prior.get('type', 'custom') != 'uniform'):
            notes.append(f'{name}: custom segmentation prior replaced by the default uniform prior '
                         '(cryoSPHERE 0.6.x cannot train with custom prior values).')
        part['segmentation_prior'] = {'type': 'uniform'}
        part.pop('segmentation_start', None)
        out[name] = part
    return out, notes


def hard_segmentation(seg_state, seg_cfg, poly):
    """Most likely segment of every residue, from saved segmenter parameters (no sampling noise)."""
    import torch
    n_res = len(poly.chain_id)
    labels = {}
    for part, cfg in seg_cfg.items():
        mu = torch.as_tensor(seg_state[f'segments_means_means.{part}']).float().reshape(-1)
        sd = torch.nn.functional.elu(torch.as_tensor(seg_state[f'segments_stds_means.{part}']).float().reshape(-1)) + 1
        logp = torch.log_softmax(torch.as_tensor(seg_state[f'segments_proportions_means.{part}']).float().reshape(-1), -1)
        if cfg.get('all_protein'):
            glob_idx = np.arange(n_res)
            pos = np.arange(n_res, dtype=np.float32)
        else:
            chain_idx = np.where(poly.chain_id == cfg['chain'])[0]
            pos = np.arange(cfg['start_res'], cfg['end_res'] + 1, dtype=np.float32)
            glob_idx = chain_idx[cfg['start_res']:cfg['end_res'] + 1]
        r = torch.as_tensor(pos)[:, None]
        score = -0.5 * (r - mu[None]) ** 2 / sd[None] ** 2 + logp[None]
        labels[part] = (glob_idx, score.argmax(-1).numpy(), int(cfg['N_segm']))
    return labels


def segment_ids(labels, n_res):
    """One integer per residue: -1 = not segmented (rigid), otherwise a global segment number."""
    out, offset = np.full(n_res, -1), 0
    for part, (idx, lab, n_segm) in labels.items():
        out[idx] = lab + offset
        offset += n_segm
    return out


MIN_RES_PER_SEGMENT = 50


def segmentation_risks(seg_cfg, poly):
    """Parts with too few residues per segment: cryoSPHERE's sampled segment widths can then underflow to 0 → NaN."""
    risks = []
    for name, part in seg_cfg.items():
        n_res = len(poly.chain_id) if part.get('all_protein') else int(part['end_res']) - int(part['start_res']) + 1
        per = n_res / int(part['N_segm'])
        if per < MIN_RES_PER_SEGMENT and 'segmentation_start_values' not in part:
            risks.append(f'{name}: {n_res} residues / {part["N_segm"]} segments = {per:.0f} per segment')
    return risks


def auto_n_segments(n_res):
    return int(min(20, max(2, n_res // MIN_RES_PER_SEGMENT)))


def first_nan_epoch(df):
    """Epoch at which the data-fit (correlation) loss became NaN, if it did."""
    if df.empty or 'Correlation loss' not in df:
        return None
    bad = df[~np.isfinite(df['Correlation loss'].astype(float))]
    return int(bad['Epoch'].iloc[0]) if len(bad) else None


NAN_HELP = ('The training loss became NaN, so the model stopped learning. With cryoSPHERE this almost always means '
            'too many segments for the number of residues (sampled segment widths underflow to zero). In 4.1, '
            f'use fewer segments (at least {MIN_RES_PER_SEGMENT} residues per segment; N_segments = 0 picks this automatically), '
            'then train a new run.')


SEG_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22',
              '#17becf', '#393b79', '#e7ba52', '#ad494a', '#637939', '#7b4173', '#3182bd', '#e6550d', '#31a354',
              '#756bb1', '#636363']


def seg_color(i):
    return '#c8c8c8' if i < 0 else SEG_COLORS[i % len(SEG_COLORS)]


# ----------------------------------------------------------------------------- volumes & FSC
def fourier_resize(vol, n_new):
    """Resample a cubic volume to n_new voxels per side by cropping/padding its Fourier transform (complex64)."""
    import scipy.fft as sfft
    n = vol.shape[0]
    if n == n_new:
        return vol.astype(np.float32)
    f = sfft.fftshift(sfft.fftn(vol.astype(np.float32), workers=-1))
    out = np.zeros((n_new,) * 3, dtype=np.complex64)
    m = min(n, n_new)
    so, sn = n // 2 - m // 2, n_new // 2 - m // 2
    out[sn:sn + m, sn:sn + m, sn:sn + m] = f[so:so + m, so:so + m, so:so + m]
    del f
    return (sfft.ifftn(sfft.ifftshift(out), workers=-1).real * (n_new / n) ** 3).astype(np.float32)


def fsc_curve(v1, v2, apix):
    """Fourier shell correlation on the half-spectrum (float32/complex64 to keep memory low for large boxes)."""
    import scipy.fft as sfft
    n = v1.shape[0]
    f1 = sfft.rfftn(v1.astype(np.float32), workers=-1)
    f2 = sfft.rfftn(v2.astype(np.float32), workers=-1)
    kz = np.fft.fftfreq(n).astype(np.float32) * n
    kx = np.arange(f1.shape[-1], dtype=np.float32)
    r = np.sqrt(kz[:, None, None] ** 2 + kz[None, :, None] ** 2 + kx[None, None, :] ** 2)
    r = np.rint(r).astype(np.int32).ravel()
    # every half-spectrum column except kx = 0 (and kx = n/2 for even n) stands for itself and its Friedel mate
    w = np.where((kx == 0) | ((n % 2 == 0) & (kx == n // 2)), 1.0, 2.0)
    w = np.broadcast_to(w[None, None, :], f1.shape).ravel()
    num = np.bincount(r, weights=w * (f1 * np.conj(f2)).real.ravel())
    d1 = np.bincount(r, weights=w * (np.abs(f1) ** 2).ravel())
    d2 = np.bincount(r, weights=w * (np.abs(f2) ** 2).ravel())
    k = n // 2
    fsc = num[:k] / np.sqrt(d1[:k] * d2[:k] + 1e-30)
    return np.arange(k) / (n * apix), fsc


def fsc_crossing(freq, fsc, threshold=0.5, sustain=3):
    """Frequency where the FSC falls below `threshold` and stays below it for `sustain` shells (ignores early dips)."""
    below = fsc < threshold
    for i in range(1, len(fsc)):
        if below[i] and below[i:i + sustain].all():
            f0, f1, c0, c1 = freq[i - 1], freq[i], fsc[i - 1], fsc[i]
            return f0 + (c0 - threshold) * (f1 - f0) / (c1 - c0) if c1 != c0 else f1
    return None


def model_mask(gmm_vol, dilate_px=3, soft_px=2.0):
    """Soft mask around the rendered model, so density the model lacks (micelle, unmodelled parts) does not bias the FSC."""
    from scipy import ndimage
    core = gmm_vol > 0.05 * gmm_vol.max()
    core = ndimage.binary_dilation(core, iterations=int(dilate_px))
    return np.clip(ndimage.gaussian_filter(core.astype(np.float32), soft_px), 0, 1)


# ----------------------------------------------------------------------------- runs, logs & checkpoints
def runs_dir():
    need('project_dir')
    d = Path(STATE['project_dir']) / 'runs'
    d.mkdir(parents=True, exist_ok=True)
    return d


def on_drive(path):
    return str(path).startswith(DRIVE_ROOT)


def run_work_dir(run_dir):
    """Where cryoSPHERE writes checkpoints (its folder_path): the run folder itself, or a local mirror for Drive projects."""
    try:
        return Path(yaml.safe_load((Path(run_dir) / 'parameters.yaml').read_text())['folder_path'])
    except Exception:
        return Path(run_dir)


def _ckpt_dirs(run_dir):
    return list(dict.fromkeys([Path(run_dir) / 'cryoSPHERE', run_work_dir(run_dir) / 'cryoSPHERE']))


def _epochs_in(d, kind):
    return {int(m.group(1)): p for p in Path(d).glob(f'{kind}*.pt') if (m := re.fullmatch(kind + r'(\d+)\.pt', p.name))}


def list_checkpoints(run_dir):
    """{epoch: (ckpt, seg)} for every epoch with both files, from the project run folder and its local work folder."""
    eps = {}
    for d in _ckpt_dirs(run_dir):                    # the local work folder comes last and wins
        ck, sg = _epochs_in(d, 'ckpt'), _epochs_in(d, 'seg')
        eps.update({e: (ck[e], sg[e]) for e in ck if e in sg})
    return dict(sorted(eps.items()))


def seg_epochs(run_dir):
    return sorted(set().union(*[set(_epochs_in(d, 'seg')) for d in _ckpt_dirs(run_dir)]))


def prune_checkpoints(run_dir, keep_last, keep_every):
    """Delete old model checkpoints (tens of MB each) in the folder cryoSPHERE writes to. Never on Google Drive, where
    deleted files only move to the Trash and keep using quota; segmentation files are tiny and always kept."""
    d = run_work_dir(run_dir) / 'cryoSPHERE'
    if not keep_last or keep_last <= 0 or on_drive(d):
        return 0
    ck, sg = _epochs_in(d, 'ckpt'), _epochs_in(d, 'seg')
    eps = sorted(e for e in ck if e in sg)
    removed = 0
    for e in eps[:-keep_last]:
        if keep_every and keep_every > 0 and (e + 1) % keep_every == 0:
            continue
        try:
            ck[e].unlink()
            removed += 1
        except FileNotFoundError:
            pass
    return removed


def sync_run(run_dir, keep_every, final=False):
    """Copy logs, segmentations, every keep_every-th checkpoint and (final=True) the latest checkpoint from the local
    work folder to the run folder on Drive. Nothing is ever deleted on Drive."""
    work = run_work_dir(run_dir)
    src, dst = work / 'cryoSPHERE', Path(run_dir) / 'cryoSPHERE'
    if work == Path(run_dir) or not src.exists():
        return
    dst.mkdir(parents=True, exist_ok=True)
    ck, sg = _epochs_in(src, 'ckpt'), _epochs_in(src, 'seg')
    eps = sorted(e for e in ck if e in sg)
    wanted = {e for e in eps if keep_every and (e + 1) % keep_every == 0} | ({eps[-1]} if final and eps else set())
    files = [sg[e] for e in eps] + [ck[e] for e in wanted] + sorted(src.glob('*.log')) + sorted(src.glob('*.yaml'))
    for f in files:
        t = dst / f.name
        try:
            if t.exists() and t.stat().st_size == f.stat().st_size and f.suffix != '.log':
                continue
            if f.suffix == '.log' and t.exists() and t.stat().st_size == f.stat().st_size:
                continue
            shutil.copyfile(f, t.with_name(t.name + '.partial'))
            os.replace(t.with_name(t.name + '.partial'), t)
        except FileNotFoundError:
            pass


_NUM = r'(-?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?|nan|-?inf)'


def parse_run_logs(run_dir):
    """Per-epoch metrics from cryoSPHERE's run.log (values are those of the first batch of each epoch)."""
    d = run_work_dir(run_dir) / 'cryoSPHERE'
    if not list(d.glob('run*.log')):
        d = Path(run_dir) / 'cryoSPHERE'
    logs = sorted(d.glob('run*.log'), key=lambda p: p.stat().st_mtime)
    rows = []
    for log in logs:
        text = log.read_text(errors='replace')
        for chunk in re.split(r'(?=Epoch: \d+ \|\|)', text)[1:]:
            chunk = re.split(r'\n\d{2}/\d{2}/\d{4} ', chunk)[0]
            row = {}
            for key, val in re.findall(r'([A-Za-z_][A-Za-z0-9_ ]*?)\s*:\s*' + _NUM, chunk):
                row[key.strip()] = float(val)
            if 'Epoch' in row:
                row['log'] = log.name
                rows.append(row)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df['Epoch'] = df['Epoch'].astype(int)
    return df.drop_duplicates(['log', 'Epoch'], keep='last').reset_index(drop=True)


LOSS_PANELS = [('Correlation loss', 'Correlation loss (data fit, lower is better)'), ('KL prior latent', 'KL prior latent'),
               ('Continuity loss', 'Continuity loss'), ('Clashing loss', 'Clashing loss'),
               ('KL prior segmentation std', 'KL segmentation std'), ('KL prior segmentation proportions', 'KL segmentation proportions')]


def plot_losses(df, title=''):
    if df.empty:
        return None
    df = df.copy()
    df['step'] = np.arange(len(df))
    panels = [(c, t) for c, t in LOSS_PANELS if c in df]
    beta_cols = [c for c in df.columns if c.endswith(' beta')]
    n = len(panels) + (1 if beta_cols else 0)
    ncol = 3
    fig, axes = plt.subplots(math.ceil(n / ncol), ncol, figsize=(15, 3.2 * math.ceil(n / ncol)), squeeze=False)
    for ax, (col, t) in zip(axes.flat, panels):
        if not np.isfinite(df[col].astype(float)).any():
            note = ' (NaN: no clashing residues, expected with the full clashing loss)' if col == 'Clashing loss' else ' (NaN)'
            ax.text(0.5, 0.5, 'all values NaN', ha='center', va='center', transform=ax.transAxes, color='gray')
            ax.set_title(t + note, fontsize=9)
            continue
        ax.plot(df['step'], df[col], lw=1, color='#1f77b4', alpha=0.45)
        if len(df) >= 5:
            ax.plot(df['step'], df[col].rolling(5, min_periods=1, center=True).mean(), lw=2, color='#1f77b4')
        ax.set_title(t, fontsize=10)
        ax.set_xlabel('epoch' if df['log'].nunique() == 1 else 'epoch (all runs in order)')
        ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
        ax.grid(alpha=0.3)
    if beta_cols:
        ax = axes.flat[len(panels)]
        for c in beta_cols:
            ax.plot(df['step'], df[c], lw=1.5, label=c.replace(' beta', ''))
        ax.set_title('Loss weights (β schedules)', fontsize=10)
        ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)
    for ax in list(axes.flat)[n:]:
        ax.axis('off')
    fig.suptitle(title or 'Training curves (first batch of each epoch)', fontsize=12)
    fig.tight_layout()
    return fig


TQDM_RE = re.compile(r'(\d+)%\|[^|]*\|\s*(\d+)/(\d+)\s*\[([^\]]*)\]')


def _fmt_secs(s):
    s = int(max(0, s))
    return f'{s // 3600}h{(s % 3600) // 60:02d}m' if s >= 3600 else f'{s // 60}m{s % 60:02d}s'


class TrainingMonitor:
    """In-place status line + loss plot for a training run; also prunes old checkpoints."""

    def __init__(self, run_dir, total_epochs, keep_last=0, keep_every=0, plot=True):
        self.run_dir, self.total = Path(run_dir), int(total_epochs)
        self.keep_last, self.keep_every, self.plot = keep_last, keep_every, plot
        self.start = time.time()
        self.done = self.epochs_done()
        self.done_t = self.start
        self.epoch_times = []
        self.progress = ''
        self.abort_reason = None
        self.status = display(HTML('<code>Starting cryoSPHERE (loading data, first epoch)…</code>'), display_id=True)
        self.fig_handle = display(HTML(''), display_id=True) if plot else None

    def epochs_done(self):
        segs = seg_epochs(self.run_dir)
        return max(segs) + 1 if segs else 0

    def on_progress(self, line):
        self.progress = line

    def update(self, force_plot=False):
        done = self.epochs_done()
        new_epoch = done > self.done
        if new_epoch:
            now = time.time()
            self.epoch_times.append((now - self.done_t) / (done - self.done))
            self.done, self.done_t = done, now
        prune_checkpoints(self.run_dir, self.keep_last, self.keep_every)
        if new_epoch:
            sync_run(self.run_dir, self.keep_every)
        eta = ''
        if self.epoch_times:
            per = float(np.median(self.epoch_times[-5:]))
            eta = f' · ~{_fmt_secs(per)}/epoch · ETA {_fmt_secs(per * max(0, self.total - done))}'
        m = TQDM_RE.search(self.progress or '')
        batch = f' · batch {m.group(2)}/{m.group(3)} [{m.group(4)}]' if m else ''
        current = f'Epoch {done + 1}/{self.total} running' if done < self.total else f'All {self.total} epochs finished'
        self.status.update(HTML(f'<code>{current} · {done} done{batch}{eta} · elapsed {_fmt_secs(time.time() - self.start)}</code>'))
        if new_epoch or force_plot:
            df = parse_run_logs(self.run_dir)
            nan_ep = first_nan_epoch(df)
            if nan_ep is not None and not force_plot:
                self.abort_reason = f'Epoch {nan_ep}: {NAN_HELP}'
            if self.plot:
                fig = plot_losses(df, f'{self.run_dir.name}: training curves (first batch of each epoch)')
                if fig is not None:
                    self.fig_handle.update(fig)
                    plt.close(fig)


class ProgressLine:
    """Shows the latest tqdm progress of a cryoSPHERE command in place."""

    def __init__(self, label):
        self.label = label
        self.handle = display(HTML(f'<code>{label}: starting…</code>'), display_id=True)

    def on_progress(self, line):
        m = TQDM_RE.search(line)
        if m:
            self.handle.update(HTML(f'<code>{self.label}: {m.group(2)}/{m.group(3)} batches [{m.group(4)}]</code>'))

    def update(self, force_plot=False):
        pass


def stream_process(cmd, log_path, cwd=None, env=None, monitor=None, echo_filter=None, show_cmd=True):
    """Run a long cryoSPHERE command: progress bars go to `monitor`, other lines are printed (filtered), all goes to log_path."""
    cmd = [str(c) for c in cmd]
    if show_cmd:
        print('$ ' + ' '.join(shlex.quote(c) for c in cmd))
    env = {**os.environ, 'PYTHONUNBUFFERED': '1', 'PYTHONWARNINGS': 'ignore', **(env or {})}
    with open(log_path, 'ab') as logf:
        logf.write(f'\n===== {time.strftime("%Y-%m-%d %H:%M:%S")} $ {" ".join(cmd)}\n'.encode())
        proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                stdin=subprocess.DEVNULL, start_new_session=True)
        buf, last_tick, tail = b'', 0.0, []
        try:
            while True:
                ready, _, _ = select.select([proc.stdout], [], [], 1.0)
                if ready:
                    chunk = os.read(proc.stdout.fileno(), 1 << 16)
                    if not chunk:
                        break
                    logf.write(chunk)
                    logf.flush()
                    buf += chunk
                    pieces = re.split(rb'[\r\n]', buf)
                    buf = pieces.pop()
                    for raw in pieces + ([buf] if TQDM_RE.search(buf.decode('utf-8', 'replace')) else []):
                        line = raw.decode('utf-8', 'replace').rstrip()
                        if not line:
                            continue
                        if TQDM_RE.search(line):
                            if monitor:
                                monitor.on_progress(line)
                            continue
                        tail = (tail + [line])[-40:]
                        if echo_filter is None or echo_filter(line):
                            print(line)
                elif proc.poll() is not None:
                    break
                if monitor and time.time() - last_tick > 3:
                    monitor.update()
                    last_tick = time.time()
                    if getattr(monitor, 'abort_reason', None):
                        kill_process_group(proc.pid)
                        fail(monitor.abort_reason)
        except KeyboardInterrupt:
            print('\nStopping cryoSPHERE …')
            kill_process_group(proc.pid)
            fail('Interrupted. Everything saved so far (checkpoints, logs) is kept.')
        finally:
            if proc.poll() is None:
                kill_process_group(proc.pid)
        proc.wait()
        if buf.strip() and not TQDM_RE.search(buf.decode('utf-8', 'replace')):
            tail.append(buf.decode('utf-8', 'replace').rstrip())
        if monitor:
            monitor.update(force_plot=True)
        return proc.returncode, tail


_NOISE = re.compile(r'(UserWarning|warnings\.warn|FutureWarning|DeprecationWarning|torch\.meshgrid|return _VF|'
                    r'TRJACTORIES|^\s*\[|^\s*\d|Saving structure)')


def quiet_filter(line):
    return not _NOISE.search(line)


# ----------------------------------------------------------------------------- analysis helpers
def analysis_config(run_dir):
    """A copy of the run's YAML files for cryosphere_analyze, so the analysis does not overwrite the training run.log."""
    cfg_dir = run_dir / 'analysis' / '_config'
    cfg_dir.mkdir(parents=True, exist_ok=True)
    params = yaml.safe_load((run_dir / 'parameters.yaml').read_text())
    missing = [f for f in (params['base_structure_path'], params['cs_star_file']['file'], params['particles_path']) if not Path(f).exists()]
    if missing:
        fail(f'The inputs this run was trained on are missing: {missing}. On a new runtime, re-run 2.1, 2.2 and 3.1 with the '
             'same data and settings so that they are recreated at the same paths.')
    params.update({'folder_path': str(cfg_dir), 'wandb': False, 'resume_training': {'model': None, 'segmentation': None}})
    write_yaml(cfg_dir / 'parameters.yaml', params)
    shutil.copy(run_dir / 'images.yaml', cfg_dir / 'images.yaml')
    return cfg_dir / 'parameters.yaml', params


def run_segmentation(run_dir, seg_path, poly):
    import torch
    params = yaml.safe_load((Path(run_dir) / 'parameters.yaml').read_text())
    state = torch.load(seg_path, map_location='cpu')
    return params['segmentation_config'], hard_segmentation(state, params['segmentation_config'], poly)


def generate_from_latents(z_sel, out_dir, names, label):
    """Decode latent vectors into structures with cryosphere_analyze --generate_structures; files are renamed to `names`."""
    run_dir, ep = Path(STATE['analysis_run']), STATE['analysis_epoch']
    eps = list_checkpoints(run_dir)
    if ep not in eps:
        fail(f'The checkpoint of epoch {ep} is no longer available; re-run 6.1 with one of the kept epochs {sorted(eps)}.')
    ckpt, seg = eps[ep]
    cfg_path, _ = analysis_config(run_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)            # never mix structures from an earlier selection into this one
    out_dir.mkdir(parents=True)
    np.save(out_dir / 'z_selected.npy', z_sel.astype(np.float32))
    ensure_gpu_free_for_cryosphere(label)
    rc, tail = stream_process(['cryosphere_analyze', '--experiment_yaml', cfg_path, '--model', ckpt, '--segmenter', seg,
                               '--output_path', out_dir, '--z', out_dir / 'z_selected.npy', '--generate_structures'],
                              out_dir / 'analyze_stdout.log', cwd=out_dir, env={'WANDB_MODE': 'disabled'}, echo_filter=quiet_filter)
    if rc != 0:
        print('\n'.join(tail)[-4000:])
        fail(f'cryosphere_analyze failed (exit code {rc}).')
    gen = out_dir / 'predicted_structures'
    paths = []
    for j, nm in enumerate(names):
        src = gen / f'structure_z_{j}.pdb'
        if not src.exists():
            fail(f'Expected {src} was not written.')
        dst = out_dir / f'{nm}.pdb'
        src.replace(dst)
        paths.append(dst)
    shutil.rmtree(gen, ignore_errors=True)
    return paths


# ----------------------------------------------------------------------------- 3D viewers
def view3d(models, colors_per_residue, chain_ids, res_ids, res_names, style='cartoon', animate=True,
           bfactors=None, width=900, height=520, labels=None):
    """models: list of (N,3) coordinate arrays. colors_per_residue: list of hex colors (len N) or None for B-factor coloring."""
    import py3Dmol
    v = py3Dmol.view(width=width, height=height)
    text = '\n'.join(ca_pdb_text(chain_ids, res_ids, res_names, xyz, bfactors, model=i + 1) for i, xyz in enumerate(models))
    if animate and len(models) > 1:
        v.addModelsAsFrames(text, 'pdb')
    elif len(models) > 1:
        v.addModels(text, 'pdb')
    else:
        v.addModel(ca_pdb_text(chain_ids, res_ids, res_names, models[0], bfactors), 'pdb')
    base = {'cartoon': {'style': 'trace', 'thickness': 0.6}} if style == 'cartoon' else {'sphere': {'scale': 0.6}}
    kind = next(iter(base))
    if colors_per_residue is None:
        hi = float(np.nanmax(bfactors)) if bfactors is not None and np.nanmax(bfactors) > 0 else 1.0
        v.setStyle({}, {kind: {**base[kind], 'colorscheme': {'prop': 'b', 'gradient': 'rwb', 'min': hi, 'max': 0}}})
    else:
        v.setStyle({}, {kind: {**base[kind], 'color': '#c8c8c8'}})
        groups = {}
        for c, r, col in zip(chain_ids, res_ids, colors_per_residue):
            groups.setdefault((str(c)[:1], col), []).append(int(r) % 10000)
        for (c, col), resi in groups.items():
            v.setStyle({'chain': c, 'resi': resi}, {kind: {**base[kind], 'color': col}})
    if animate and len(models) > 1:
        v.animate({'loop': 'backAndForth', 'interval': 250})
    v.zoomTo()
    return v


CSH_LOADED = True
ok(f'Helpers loaded{" (Colab)" if IN_COLAB else ""}.')

In [ ]:
#@title 1.3 · Install cryoSPHERE { display-mode: "form" }
#@markdown Installs the `cryosphere` package and its command-line tools (about 2–4 minutes). Colab's preinstalled PyTorch is kept.
install_from = "PyPI (recommended)"  #@param ["PyPI (recommended)", "GitHub repository"]
#@markdown PyPI version (empty = latest release):
pypi_version = ""  #@param {type:"string"}
#@markdown GitHub repository and branch/tag/commit (only used for *GitHub repository*; any fork works):
github_repo = "https://github.com/gabriel-ducrocq/cryoSPHERE"  #@param {type:"string"}
github_ref = "main"  #@param {type:"string"}

def _main():
    import importlib.metadata as md
    def version(pkg):
        try:
            return md.version(pkg)
        except md.PackageNotFoundError:
            return None
    numpy_before = version('numpy')
    if install_from.startswith('GitHub'):
        repo = github_repo.strip().rstrip('/')
        repo = repo if repo.endswith('.git') else repo + '.git'
        spec = f'git+{repo}@{github_ref.strip() or "main"}'
    else:
        spec = f'cryosphere=={pypi_version.strip()}' if pypi_version.strip() else 'cryosphere'
    print('Installing … (pip may print dependency-resolver warnings about other Colab packages; they do not affect cryoSPHERE)')
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '--progress-bar', 'off', spec, 'py3Dmol'])
    importlib.invalidate_caches()
    missing = [t for t in CS_TOOLS if shutil.which(t) is None]
    if missing:
        fail(f'Installation finished but these commands are missing: {missing}')
    code_, out = run_cmd(['cryosphere_train', '--help'], echo=False, check=False)
    if code_ != 0:
        print(out[-3000:])
        fail('cryosphere_train does not start (see the error above).')
    ok(f'cryoSPHERE {version("cryosphere")} installed (numpy {version("numpy")}, torch {version("torch")}, '
       f'biotite {version("biotite")}, wandb {version("wandb")}).')
    if numpy_before and version('numpy') != numpy_before and 'numpy' in sys.modules:
        warn(f'numpy changed from {numpy_before} to {version("numpy")}. Restarting the runtime so the notebook picks it up; '
             'afterwards run 1.2 and continue with 1.4.')
        time.sleep(2)
        os.kill(os.getpid(), signal.SIGKILL)

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

In [ ]:
#@title 1.4 · Choose where to keep your work { display-mode: "form" }
#@markdown *Colab disk* is fast but erased when the runtime ends. *Google Drive* keeps configs, checkpoints and results across sessions
#@markdown (in `MyDrive/cryoSPHERE/<project_name>`); training data are still read from the fast local disk (see 2.1).
storage = "Colab disk (temporary)"  #@param ["Colab disk (temporary)", "Google Drive (persistent)"]
project_name = "my_cryosphere_project"  #@param {type:"string"}

def _main():
    name = re.sub(r'[^\w.\-]+', '_', project_name.strip()) or 'my_cryosphere_project'
    if storage.startswith('Google Drive'):
        if not IN_COLAB:
            fail('Google Drive is only available inside Colab.')
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            drive.mount('/content/drive')
        base = Path('/content/drive/MyDrive/cryoSPHERE')
    else:
        base = Path('/content/cryoSPHERE' if os.path.exists('/content') else Path.cwd() / 'cryoSPHERE')
    pdir = base / name
    (pdir / 'runs').mkdir(parents=True, exist_ok=True)
    if load_state(pdir):
        STATE['project_dir'] = str(pdir)
        ok(f'Re-opened project {pdir}')
        for key, label in [('data_dir', 'data'), ('particles_file', 'particles'), ('structure_file', 'structure'), ('run_dir', 'current run')]:
            if STATE.get(key):
                print(f'   {label:12s} {STATE[key]}')
        todo = []
        if not shutil.which('cryosphere_train'):
            todo.append('1.3 (install cryoSPHERE)')
        gone = [k for k in ('data_dir', 'metadata', 'structure_file') if STATE.get(k) and not Path(STATE[k]).exists()]
        if gone:
            todo.append('2.1, 2.2 and 3.1 with the same settings (the data on the Colab disk is gone)')
        elif STATE.get('structure_centered') and not Path(STATE['structure_centered']).exists():
            todo.append('3.1')
        if todo:
            warn('This looks like a new runtime. Before continuing, re-run ' + '; then '.join(todo) + '. '
                 'Your runs, checkpoints and settings in the project folder are kept.')
    else:
        STATE.clear()
        STATE.update({'project_dir': str(pdir), 'storage': storage, 'wandb': False})
        ok(f'New project folder: {pdir}')
    save_state()

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

In [ ]:
#@title 1.5 · (Optional) Log training to Weights & Biases { display-mode: "form" }
#@markdown cryoSPHERE always writes `run.log`, and this notebook plots it, so W&B is optional.
#@markdown To use it, add a Colab secret named `WANDB_API_KEY` (🔑 icon in the left sidebar, enable notebook access) and choose **Online**.
wandb_mode = "Off"  #@param ["Off", "Online"]
wandb_project = "cryoSPHERE"  #@param {type:"string"}

def _main():
    need('project_dir')
    if wandb_mode == 'Off':
        STATE['wandb'] = False
        save_state()
        ok('W&B logging off. Training metrics go to run.log and are plotted in 5.2.')
        return
    key = os.environ.get('WANDB_API_KEY', '')
    if not key and IN_COLAB:
        try:
            from google.colab import userdata
            key = userdata.get('WANDB_API_KEY') or ''
        except Exception as e:
            info(f'No usable Colab secret WANDB_API_KEY ({type(e).__name__}).')
    if not key:
        from getpass import getpass
        key = getpass('Paste your W&B API key (https://wandb.ai/authorize): ').strip()
    if not key:
        fail('No W&B API key given.')
    os.environ['WANDB_API_KEY'] = key       # kept in memory only, never written to the project folder
    if len(key) != 40:
        info('This is a new-style W&B key; upgrading wandb because the version pinned by cryoSPHERE only accepts 40-character keys.')
        run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '--progress-bar', 'off', '-U', 'wandb'])
    code_, out = run_cmd([sys.executable, '-c', 'import wandb; print(wandb.login(verify=True))'], echo=False, check=False)
    if code_ != 0 or 'True' not in out:
        print(out[-2000:])
        fail('W&B rejected the API key.')
    STATE['wandb'], STATE['wandb_project'] = True, wandb_project.strip() or 'cryoSPHERE'
    save_state()
    ok(f'W&B ready: runs will be logged to project "{STATE["wandb_project"]}".')

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

---
## 2 · Get the data

cryoSPHERE needs the particle stack(s), a metadata file with poses and CTF from a consensus refinement, and an atomic model fitted into the consensus map.
The **example dataset** (Zenodo [15794610](https://zenodo.org/records/15794610), about 3 GB) has 20,000 simulated images of a protein whose conformations come from molecular dynamics; it is the quickest way to see the whole pipeline.

Large files reach Colab fastest through Google Drive: upload once to Drive, then pick *Google Drive folder*. Direct upload from your computer is fine for small datasets.

In [ ]:
#@title 2.1 · Download or locate the data { display-mode: "form" }
data_source = "Zenodo: cryoSPHERE example dataset"  #@param ["Zenodo: cryoSPHERE example dataset", "Zenodo: another record", "Google Drive folder", "Upload from my computer", "Download from a URL"]
#@markdown **Zenodo: another record**: record ID (the number at the end of the record's URL):
zenodo_record = ""  #@param {type:"string"}
#@markdown **Google Drive folder**: path inside `MyDrive` (e.g. `cryoem/my_dataset`), or a full path such as `/content/drive/Shareddrives/Lab/dataset`:
drive_folder = ""  #@param {type:"string"}
#@markdown **URL**: direct link to a file or an archive (`.zip`, `.tar.gz`, …), which is unpacked:
url = ""  #@param {type:"string"}
#@markdown Copy Drive data to the local disk first (much faster training; needs enough free disk):
copy_to_local_disk = True  #@param {type:"boolean"}

def _main():
    need('project_dir')
    local_root = Path('/content/cryosphere_data' if os.path.exists('/content') else Path(STATE['project_dir']) / 'data')
    if data_source.startswith('Zenodo'):
        rec = '15794610' if 'example' in data_source else re.sub(r'\D', '', zenodo_record)
        if not rec:
            fail('Enter a Zenodo record ID.')
        title, files = zenodo_files(rec)
        data_dir = local_root / f'zenodo_{rec}'
        print(f'Zenodo record {rec}: {title}')
        show_table([{'file': f['name'], 'size': human_size(f['size'])} for f in files])
        total = sum(f['size'] for f in files if not (data_dir / f['name']).exists())
        if total * 2.2 > disk_free():
            fail(f'Not enough free disk: need about {human_size(total * 2.2)} (download + unpacking), have {human_size(disk_free())}.')
        for f in files:
            download(f['url'], data_dir / f['name'], expected_size=f['size'], md5=f['md5'])
    elif data_source == 'Google Drive folder':
        if not IN_COLAB:
            fail('Google Drive is only available inside Colab.')
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            drive.mount('/content/drive')
        folder = drive_folder.strip()
        if folder.startswith('/'):
            src = Path(folder)
        else:
            src = Path('/content/drive/MyDrive') / re.sub(r'^(My ?Drive/)', '', folder.strip('/'))
        if not folder or not src.is_dir():
            fail(f'Folder not found in your Drive: {src}')
        if copy_to_local_disk:
            data_dir = local_root / f'{src.name}_{hashlib.md5(str(src).encode()).hexdigest()[:6]}'
            size = sum(p.stat().st_size for p in src.rglob('*') if p.is_file())
            have = sum(p.stat().st_size for p in data_dir.rglob('*') if p.is_file()) if data_dir.exists() else 0
            if (size - have) * 1.1 > disk_free():
                fail(f'The folder ({human_size(size)}) does not fit on the local disk; untick copy_to_local_disk.')
            info(f'Copying {human_size(size)} from Drive to the local disk …')
            data_dir.mkdir(parents=True, exist_ok=True)
            if shutil.which('rsync'):
                run_cmd(['rsync', '-a', f'{src}/', f'{data_dir}/'], echo=False)
            else:
                shutil.copytree(src, data_dir, dirs_exist_ok=True)
        else:
            data_dir = src
    elif data_source == 'Upload from my computer':
        if not IN_COLAB:
            fail('Uploading from the browser is only available inside Colab.')
        from google.colab import files
        info('Choose your files (several at once is fine). Archives are unpacked.')
        uploaded = files.upload()
        if not uploaded:
            fail('No file was uploaded.')
        # same files → same folder, so a re-upload on a new runtime restores the paths earlier runs were trained on
        sig = hashlib.md5(json.dumps(sorted((n, len(b)) for n, b in uploaded.items())).encode()).hexdigest()[:8]
        data_dir = local_root / f'upload_{sig}'
        data_dir.mkdir(parents=True, exist_ok=True)
        for name in uploaded:
            shutil.move(name, data_dir / name)
    else:
        if not url.strip():
            fail('Enter a URL.')
        name = url.strip().split('?')[0].rstrip('/').split('/')[-1] or 'download'
        stem = re.sub(r'(\.tar)?\.(zip|tar|tgz|gz|bz2|xz)$', '', name)
        data_dir = local_root / f'{stem}_{hashlib.md5(url.strip().encode()).hexdigest()[:6]}'
        size = remote_size(url.strip())
        if size and not (data_dir / name).exists() and size * (2.2 if is_archive(name) else 1.1) > disk_free():
            fail(f'Not enough free disk for {name} ({human_size(size)}{", plus unpacking" if is_archive(name) else ""}).')
        download(url.strip(), data_dir / name, expected_size=size or None)
    archives = [p for p in Path(data_dir).rglob('*') if p.is_file() and is_archive(p) and not p.name.startswith('.')]
    if archives and str(data_dir).startswith('/content/drive'):
        fail(f'{archives[0].name} is an archive inside your Drive folder. Tick copy_to_local_disk so it is unpacked on the '
             'local disk (unpacking inside Drive would use your Drive quota and be very slow), or unpack it yourself first.')
    for arc in archives:
        extract(arc, arc.parent)
    files_ = [p for p in Path(data_dir).rglob('*') if p.is_file() and not p.name.startswith('.')]
    rows = [{'file': str(p.relative_to(data_dir)), 'size': human_size(p.stat().st_size)} for p in sorted(files_)[:40]]
    show_table(rows)
    if len(files_) > 40:
        info(f'… and {len(files_) - 40} more files.')
    if STATE.get('data_dir') != str(data_dir):
        for k in ('particles_file', 'structure_file', 'map_file', 'dataset', 'structure_centered', 'map_centered', 'lp_fsc', 'centering_shift', 'inputs_signature'):
            STATE.pop(k, None)
    STATE['data_dir'] = str(data_dir)
    save_state()
    ok(f'Data folder: {data_dir}')

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

In [ ]:
#@title 2.2 · Identify, validate and fix the input files { display-mode: "form" }
#@markdown Leave fields empty to auto-detect. Paths can be absolute or relative to the data folder.
#@markdown The checks catch problems that would otherwise crash cryoSPHERE (or silently give NaN losses) hours into training, and fix them in a copy of your metadata where possible.
particles_file = ""  #@param {type:"string"}
#@markdown ↳ RELION `.star` or cryoSPARC `.cs` with the poses and CTF of every particle.
structure_file = ""  #@param {type:"string"}
#@markdown ↳ atomic model (`.pdb` or `.cif`) fitted into the consensus map.
consensus_map_file = ""  #@param {type:"string"}
#@markdown ↳ optional consensus map (`.mrc`), for centering and the fit check. Type `none` to ignore any map.
particles_folder = ""  #@param {type:"string"}
#@markdown ↳ folder that the image paths inside the metadata are relative to.
#@markdown **cryoSPARC `.cs` only**
cs_passthrough_file = ""  #@param {type:"string"}
#@markdown ↳ `*_passthrough_particles.cs`; merged in when CTF or image fields are missing from the main file (auto-detected when it sits next to it).
cs_ab_initio = "auto"  #@param ["auto", "yes", "no"]
cs_heterogeneous_refinement = False  #@param {type:"boolean"}

def _resolve(p, data_dir):
    p = p.strip()
    if not p:
        return None
    q = Path(p) if os.path.isabs(p) else Path(data_dir) / p
    if not q.exists():
        fail(f'File not found: {q}')
    return q

def _pick_metadata(data_dir):
    """The particle metadata with poses: final outputs first, otherwise the latest iteration."""
    valid = []
    for p in find_files(data_dir, ('.star', '.cs')):
        if 'passthrough' in p.name.lower():
            continue
        try:
            if p.suffix == '.star':
                _, parts = read_star(p)
                if {'rlnImageName', 'rlnAngleRot', 'rlnAngleTilt', 'rlnAnglePsi'} <= set(parts):
                    valid.append(p)
            elif cs_alignment_key(read_cs(p)):
                valid.append(p)
        except Exception:
            continue
    if not valid:
        return None
    def key(p):
        n = p.name.lower()
        name_score = -sum(k in n for k in ('particles', 'data')) + sum(k in n for k in ('model', 'optimiser', 'sampling', 'half', 'class'))
        it = re.findall(r'(?:_it|_)(\d{3,})', p.stem)
        return (name_score, -1e9 if not it else -int(it[-1]), -p.stat().st_mtime, len(p.parts))
    valid.sort(key=key)
    if len(valid) > 1:
        others = ', '.join(str(p.relative_to(data_dir)) for p in valid[1:6])
        info(f'{len(valid)} metadata files with poses found; using {valid[0].relative_to(data_dir)} '
             f'(others: {others}{" …" if len(valid) > 6 else ""}). Set particles_file to choose another.')
    return valid[0]

def _is_volume(p):
    try:
        shape, _ = stack_header(p)
        return len(shape) == 3 and shape[0] == shape[1] == shape[2] and shape[0] > 1
    except Exception:
        return False

def _main():
    need('data_dir')
    data_dir = Path(STATE['data_dir'])
    work = data_dir / 'cryosphere_inputs'
    work.mkdir(exist_ok=True)
    # ---------- locate files
    meta = _resolve(particles_file, data_dir) or _pick_metadata(data_dir)
    if meta is None:
        fail('No particle metadata (.star with rlnImageName, or cryoSPARC .cs) found. Fill in particles_file.')
    pdbs = rank_by_name(find_files(data_dir, ('.pdb', '.cif', '.mmcif', '.ent')), prefer=('centered', 'base_structure', 'fit'),
                        avoid=('uncentered', 'structure_z', 'predicted', 'alphafold', 'af-'))
    struct = _resolve(structure_file, data_dir) or next(iter(pdbs), None)
    if struct is None:
        fail('No atomic model (.pdb/.cif) found. Fill in structure_file.')
    if not structure_file.strip() and len(pdbs) > 1:
        info(f'{len(pdbs)} atomic models found; using {struct.relative_to(data_dir)} (others: '
             f'{", ".join(str(p.relative_to(data_dir)) for p in pdbs[1:5])}). Set structure_file to choose another.')
    if consensus_map_file.strip().lower() == 'none':
        cmap = None
    else:
        maps = rank_by_name([p for p in find_files(data_dir, ('.mrc', '.map')) if _is_volume(p)],
                            prefer=('consensus', 'map', 'refine', 'volume'), avoid=('half', 'mask', 'gmm', 'local'))
        cmap = _resolve(consensus_map_file, data_dir) or next(iter(maps), None)
        if not consensus_map_file.strip() and len(maps) > 1:
            info(f'{len(maps)} maps found; using {cmap.relative_to(data_dir)} (others: '
                 f'{", ".join(str(p.relative_to(data_dir)) for p in maps[1:5])}). Set consensus_map_file to choose another.')
    changes = []
    # ---------- metadata
    if meta.suffix == '.star':
        kind = 'star'
        optics, parts = read_star(meta)
        parts = parts.copy()
        miss = [c for c in STAR_REQUIRED if c not in parts]
        if miss:
            fail(f'{meta.name} lacks required columns {miss}. cryoSPHERE needs poses and CTF from a consensus refinement.')
        for c in STAR_OPTICS:
            if (optics is None or c not in optics) and c not in parts:
                fail(f'{meta.name} has no {c} (neither in data_optics nor in data_particles).')
        if not ({'rlnOriginXAngst', 'rlnOriginYAngst'} <= set(parts) or {'rlnOriginX', 'rlnOriginY'} <= set(parts)):
            parts['rlnOriginXAngst'] = 0.0
            parts['rlnOriginYAngst'] = 0.0
            changes.append('no particle shifts found: set rlnOriginXAngst/YAngst to 0')
        if 'rlnPhaseShift' not in parts:
            parts['rlnPhaseShift'] = 0.0
            changes.append('added rlnPhaseShift = 0 (without it cryoSPHERE computes a NaN CTF)')
        if optics is not None and len(optics) > 1:
            cols = [c for c in ('rlnVoltage', 'rlnSphericalAberration', 'rlnAmplitudeContrast', 'rlnImagePixelSize', 'rlnImageSize') if c in optics]
            if any(optics[c].nunique() > 1 for c in cols):
                fail(f'{meta.name} has {len(optics)} optics groups with different settings; cryoSPHERE supports one. '
                     'Split the particles by optics group (e.g. relion_star_handler) and train on one group.')
            first = optics.iloc[[0]].copy()
            if 'rlnOpticsGroup' in parts:
                parts['rlnOpticsGroup'] = first['rlnOpticsGroup'].iloc[0]
            optics = first.reset_index(drop=True)
            changes.append('merged identical optics groups into one')
        apix = None
        if optics is not None and 'rlnImagePixelSize' in optics:
            apix = float(optics['rlnImagePixelSize'].iloc[0])
        elif 'rlnImagePixelSize' in parts:
            apix = float(parts['rlnImagePixelSize'].iloc[0])
        elif {'rlnDetectorPixelSize', 'rlnMagnification'} <= set(parts):     # RELION 3.0 style
            apix = float(parts['rlnDetectorPixelSize'].iloc[0]) * 1e4 / float(parts['rlnMagnification'].iloc[0])
        box = int(optics['rlnImageSize'].iloc[0]) if optics is not None and 'rlnImageSize' in optics else None
        refs = image_refs('star', parts)
        n = len(parts)
        abinit = hetrefine = False
    else:
        kind = 'cs'
        arr = read_cs(meta)
        pt = _resolve(cs_passthrough_file, data_dir)
        need_fields = CS_CTF + CS_BLOB + ['blob/shape', 'blob/psize_A']
        if pt is None and any(f not in arr.dtype.names for f in need_fields):
            cands = [p for p in meta.parent.glob('*passthrough*.cs')]
            pt = cands[0] if len(cands) == 1 else None
        if pt is not None:
            arr = merge_cs_passthrough(arr, read_cs(pt))
            changes.append(f'merged fields from {pt.name}')
        miss = [f for f in need_fields if f not in arr.dtype.names]
        if miss:
            fail(f'{meta.name} lacks fields {miss}. Provide the matching passthrough file (cs_passthrough_file), '
                 'or export the particles from cryoSPARC with poses, CTF and blob information.')
        key = cs_alignment_key(arr)
        if key is None:
            fail(f'{meta.name} has no alignments (alignments3D/pose or alignments_class_0/pose).')
        abinit = {'auto': key == 'alignments_class_0', 'yes': True, 'no': False}[cs_ab_initio]
        if f'{"alignments_class_0" if abinit else "alignments3D"}/pose' not in arr.dtype.names:
            fail(f'cs_ab_initio={cs_ab_initio} does not match the file (it has {key}/pose).')
        hetrefine = bool(cs_heterogeneous_refinement)
        apix = float(np.asarray(arr['blob/psize_A']).ravel()[0])
        box = int(np.asarray(arr['blob/shape'])[0][0])
        akey = 'alignments_class_0' if abinit else 'alignments3D'
        if f'{akey}/psize_A' in arr.dtype.names:
            align_psize = float(np.asarray(arr[f'{akey}/psize_A']).ravel()[0])
            if align_psize > 0 and abs(align_psize / apix - 1) > 0.01:
                arr[f'{akey}/shift'] = arr[f'{akey}/shift'] * (align_psize / apix)
                changes.append(f'converted {akey}/shift from the alignment pixel size ({align_psize:.3f} Å) '
                               f'to the particle pixel size ({apix:.3f} Å)')
            if hetrefine:
                info('cs_heterogeneous_refinement ignored: the file records the pixel size of its shifts, so no 2× guess is needed.')
            hetrefine = False
        elif abinit or hetrefine:
            warn('The file does not record the pixel size used for alignment. If the poses come from a job that aligned '
                 'downsampled images (ab-initio, heterogeneous refinement), the shifts may be scaled wrongly; '
                 'cs_heterogeneous_refinement assumes 2× downsampling.')
        refs = image_refs('cs', arr)
        n = len(arr)
        parts = arr
    if n < 100:
        fail(f'Only {n} particles: cryoSPHERE needs at least 100 (it estimates the image normalisation on 100 of them).')
    # ---------- image stacks
    user_folder = _resolve(particles_folder, data_dir) if particles_folder.strip() else None
    cands = [user_folder, meta.parent, data_dir, meta.parent.parent, meta.parent.parent.parent]
    folder = resolve_particles_folder(refs, [str(c) for c in cands if c])
    if folder is None:
        mapping, problem = relocate_stacks(refs, data_dir)
        if mapping is None:
            fail(f'Cannot match the image stacks referenced in {meta.name} to files in the data folder: {problem}. '
                 'Put the .mrcs files in the data folder or set particles_folder.')
        rel = {}
        for k, (orig, found) in enumerate(sorted(mapping.items())):
            r = os.path.relpath(found, data_dir)
            if not r.isascii():            # cryoSPHERE decodes cryoSPARC paths as ASCII
                (work / 'stacks').mkdir(exist_ok=True)
                link = work / 'stacks' / f'stack_{k:05d}{Path(found).suffix}'
                if link.is_symlink() or link.exists():
                    link.unlink()
                link.symlink_to(found)
                r = os.path.relpath(link, data_dir)
            rel[orig] = r
        if kind == 'star':
            parts['rlnImageName'] = [f'{i + 1:06d}@{rel[p]}' for i, p in refs]
        else:
            new = np.array([rel[p].encode('ascii') for _, p in refs])
            descr = [('blob/path', f'S{new.dtype.itemsize}') if d[0] == 'blob/path' else d for d in parts.dtype.descr if d[0]]
            fixed = np.zeros(len(parts), dtype=np.dtype(descr))
            for nm in parts.dtype.names:
                fixed[nm] = new if nm == 'blob/path' else parts[nm]
            parts = fixed
        folder = str(data_dir)
        refs = image_refs(kind, parts)
        changes.append('rewrote image paths to the stacks found in the data folder')
    missing = missing_stacks(refs, folder)
    if missing:
        fail(f'{len(missing)} of the {len({p for _, p in refs})} image stacks referenced in {meta.name} are missing '
             f'(e.g. {", ".join(missing[:3])}). Check that the upload/copy is complete.')
    first_stack = os.path.join(folder, refs[0][1])
    shape, voxel = stack_header(first_stack)
    if box is None:
        box = shape[-1]
    if not apix:
        _, data_images = next(((p, lenient_yaml_load(p)) for p in find_files(data_dir, ('images.yaml', 'images.yml'))), (None, {}))
        if data_images.get('apix') and int(data_images.get('Npix', -1)) == int(box):
            apix = float(data_images['apix'])
            info(f'Pixel size {apix} Å taken from the data folder\'s images.yaml (the metadata does not state it).')
        elif voxel and voxel > 0:
            apix = voxel
            warn(f'Pixel size {apix:.4f} Å taken from the image stack header; check that it is right.')
        else:
            fail('Could not determine the pixel size: add rlnImagePixelSize to the metadata or an images.yaml with apix.')
    if shape[-1] != box or shape[-2] != box:
        fail(f'Images in {Path(first_stack).name} are {shape[-2]}×{shape[-1]} px but the metadata says {box} px.')
    if voxel and abs(voxel - apix) / apix > 0.01:
        warn(f'Pixel size: stack header says {voxel:.4f} Å, metadata says {apix:.4f} Å; using the metadata value.')
    nframes = shape[0] if len(shape) == 3 else 1
    bad = [i for i, p in refs[:200] if p == refs[0][1] and i >= nframes]
    if bad:
        fail(f'Image index {bad[0] + 1} is beyond the {nframes} images in {Path(first_stack).name}.')
    # ---------- write the prepared metadata
    if changes:
        out = work / f'{meta.stem}_cryosphere{meta.suffix}'
        if kind == 'star':
            import starfile
            starfile.write({'optics': optics, 'particles': parts} if optics is not None else parts, out, overwrite=True)
        else:
            save_cs(parts, out)
        meta_used = out
    else:
        meta_used = meta
    # ---------- structure
    if struct.suffix in ('.ent', '.PDB', '.Pdb') or (struct.suffix.lower() == '.pdb' and struct.suffix != '.pdb'):
        pdb_out = work / f'{struct.stem}.pdb'
        shutil.copyfile(struct, pdb_out)      # cryosphere_center_origin only accepts the exact extension .pdb
        struct = pdb_out
    if struct.suffix.lower() in ('.cif', '.mmcif'):
        import gemmi
        pdb_out = work / f'{struct.stem}.pdb'
        st = gemmi.read_structure(str(struct))
        st.setup_entities()
        st.shorten_chain_names()
        st.write_pdb(str(pdb_out))
        changes.append(f'converted {struct.name} to PDB format')
        struct = pdb_out
    poly = load_polymer(struct)
    if len(poly.chain_id) == 0:
        fail(f'No amino-acid C-alpha (or nucleotide C1\') atoms found in {struct.name}.')
    keys = pd.Series([f'{c}:{r}' for c, r in zip(poly.chain_id, poly.res_id)])
    if keys.duplicated().any():        # insertion codes (e.g. 52A) would crash cryoSPHERE's continuity loss
        dup = ', '.join(keys[keys.duplicated()].unique()[:5])
        pdb_out = work / f'{struct.stem}_renumbered.pdb'
        renumber_pdb(struct, pdb_out)
        changes.append(f'some residue numbers repeat ({dup}; insertion codes): residues after them were renumbered '
                       '(gaps kept). Use the numbers in the table below for custom segmentation parts in 4.1')
        struct = pdb_out
        poly = load_polymer(struct)
    # ---------- summary
    ok(f'Particles: {n:,} images of {box}×{box} px at {apix:.3f} Å/px ({kind} file {meta.name})')
    ok(f'Image stacks found in {folder}')
    ok(f'Structure: {struct.name}, {len(poly.chain_id):,} residues in {poly.num_chains} chain(s)')
    show_table(chain_table(poly))
    if cmap is not None:
        mshape, mvox = stack_header(cmap)
        ok(f'Consensus map: {cmap.name}, {mshape[-1]} px at {mvox:.3f} Å/px')
    else:
        info('No consensus map: centering will use the particle box size, and the fit check (3.2) is skipped.')
    for c in changes:
        info(f'Fixed: {c}')
    if changes:
        info(f'Prepared metadata written to {meta_used} (the original is untouched).')
    if kind == 'cs':
        info(f'cryoSPARC options: ab-initio poses = {abinit}, heterogeneous refinement = {hetrefine}.')
    STATE.update({'particles_file': str(meta), 'metadata': str(meta_used), 'kind': kind, 'particles_folder': folder,
                  'structure_file': str(struct), 'map_file': str(cmap) if cmap else None,
                  'dataset': {'n': int(n), 'box': int(box), 'apix': float(apix), 'abinit': bool(abinit), 'hetrefine': bool(hetrefine),
                              'n_residues': int(len(poly.chain_id)), 'chains': [str(c) for c in pd.unique(poly.chain_id)]}})
    sig = [str(struct), str(cmap) if cmap else None, int(box), round(float(apix), 4)]
    if STATE.get('inputs_signature') != sig:     # centering / FSC results belong to the previous inputs
        for k in ('structure_centered', 'map_centered', 'lp_fsc', 'centering_shift'):
            STATE.pop(k, None)
    STATE['inputs_signature'] = sig
    save_state()

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

In [ ]:
#@title 2.3 · (Optional) Look at the data { display-mode: "form" }
#@markdown Random particles (raw and low-pass filtered), the defocus distribution, the viewing directions and the atomic model.
n_particles_to_show = 16  #@param {type:"slider", min:4, max:32, step:4}

def _main():
    need('dataset')
    from scipy.ndimage import gaussian_filter
    ds, kind = STATE['dataset'], STATE['kind']
    meta = Path(STATE['metadata'])
    if kind == 'star':
        _, parts = read_star(meta)
        refs = image_refs('star', parts)
        dfu = parts['rlnDefocusU'].to_numpy()
        from roma import euler_to_rotmat
        import torch
        ang = np.radians(parts[['rlnAngleRot', 'rlnAngleTilt', 'rlnAnglePsi']].to_numpy(dtype=np.float32))
        R = euler_to_rotmat('ZYZ', torch.tensor(ang)).numpy()
    else:
        arr = read_cs(meta)
        refs = image_refs('cs', arr)
        dfu = np.asarray(arr['ctf/df1_A'])
        from roma import rotvec_to_rotmat
        import torch
        key = 'alignments_class_0' if ds['abinit'] else 'alignments3D'
        R = np.transpose(rotvec_to_rotmat(torch.tensor(np.asarray(arr[f'{key}/pose'], dtype=np.float32))).numpy(), (0, 2, 1))
    rng = np.random.default_rng(0)
    idx = np.sort(rng.choice(ds['n'], size=min(n_particles_to_show, ds['n']), replace=False))
    imgs = [read_particle_image(os.path.join(STATE['particles_folder'], refs[i][1]), refs[i][0]) for i in idx]
    ncol = min(8, len(imgs))
    nrow = math.ceil(len(imgs) / ncol)
    fig, axes = plt.subplots(2 * nrow, ncol, figsize=(1.6 * ncol, 3.3 * nrow), squeeze=False)
    for k, (i, im) in enumerate(zip(idx, imgs)):
        r, c = divmod(k, ncol)
        for row, data in ((2 * r, im), (2 * r + 1, gaussian_filter(im, 2.0))):
            axes[row, c].imshow(data, cmap='gray')
            axes[row, c].set_xticks([]); axes[row, c].set_yticks([])
        axes[2 * r, c].set_title(f'#{i}', fontsize=8)
    for ax in axes.flat[len(imgs) * 2:]:
        ax.axis('off')
    fig.suptitle('Random particles: raw (upper) and low-pass filtered (lower)', fontsize=11)
    fig.tight_layout()
    plt.show()
    view_dirs = R[:, 2, :] if kind == 'cs' else np.transpose(R, (0, 2, 1))[:, 2, :]
    az = np.degrees(np.arctan2(view_dirs[:, 1], view_dirs[:, 0]))
    el = np.degrees(np.arcsin(np.clip(view_dirs[:, 2], -1, 1)))
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
    axes[0].hist(dfu / 1e4, bins=60, color='#1f77b4')
    axes[0].set_xlabel('defocus U (µm)'); axes[0].set_ylabel('particles'); axes[0].set_title('Defocus')
    hb = axes[1].hexbin(az, el, gridsize=(45, 20), cmap='viridis', mincnt=1)
    axes[1].set_xlabel('azimuth (°)'); axes[1].set_ylabel('elevation (°)'); axes[1].set_title('Viewing directions')
    fig.colorbar(hb, ax=axes[1], label='particles')
    fig.tight_layout()
    plt.show()
    poly = load_polymer(STATE.get('structure_centered') or STATE['structure_file'])
    chains = list(pd.unique(poly.chain_id))
    colors = [SEG_COLORS[chains.index(c) % len(SEG_COLORS)] for c in poly.chain_id]
    extent = np.linalg.norm(poly.coord - poly.coord.mean(0), axis=1).max()
    info(f'Model radius {extent:.0f} Å; particle box {ds["box"] * ds["apix"]:.0f} Å wide. Colored by chain ({", ".join(chains)}).')
    view3d([poly.coord], colors, poly.chain_id, poly.res_id, poly.res_name, animate=False, height=420).show()

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

---
## 3 · Prepare the atomic model

cryoSPHERE places the model in a frame whose origin is the **center of the particle box**. A model fitted in ChimeraX sits in the map's frame instead (origin at a corner), so it has to be shifted once. Step 3.1 does that automatically when needed and 3.2 checks that model, map and particles line up.

In [ ]:
#@title 3.1 · Center the model (and map) on the box { display-mode: "form" }
centering = "Auto"  #@param ["Auto", "cryosphere_center_origin (uses the consensus map)", "Shift by half the particle box (no map needed)", "Skip: the model is already centered"]
#@markdown *Auto* skips centering when the model is already near the origin, otherwise uses the map if there is one.

def _map_origin(path):
    import mrcfile
    with mrcfile.open(path, permissive=True, header_only=True) as m:
        h = m.header
        org = np.array([float(h.origin.x), float(h.origin.y), float(h.origin.z)])
        start = np.array([int(h.nxstart), int(h.nystart), int(h.nzstart)], dtype=float)
        vox = np.array([float(m.voxel_size.x), float(m.voxel_size.y), float(m.voxel_size.z)])
        n = np.array([int(h.nx), int(h.ny), int(h.nz)])
    return org, start, vox, n

def _main():
    need('dataset')
    ds = STATE['dataset']
    box_A = ds['box'] * ds['apix']
    src = Path(STATE['structure_file'])
    poly = load_polymer(src)
    centroid = poly.coord.mean(0)
    already = np.linalg.norm(centroid) < 0.25 * box_A
    info(f'Model centroid ({centroid[0]:.1f}, {centroid[1]:.1f}, {centroid[2]:.1f}) Å; box {box_A:.0f} Å.')
    cmap = STATE.get('map_file')
    mode = centering
    if mode == 'Auto':
        mode = 'skip' if already else ('map' if cmap else 'box')
    elif mode.startswith('cryosphere_center_origin'):
        mode = 'map'
        if not cmap:
            fail('No consensus map was found in 2.2; choose "Shift by half the particle box" instead.')
    elif mode.startswith('Shift'):
        mode = 'box'
    else:
        mode = 'skip'
    out_dir = Path(STATE['data_dir']) / 'centered'
    out_dir.mkdir(exist_ok=True)
    STATE['map_centered'] = None
    shift = np.zeros(3)
    if mode == 'skip':
        if not already:
            warn('The model is far from the box center but centering was skipped; make sure this is intended.')
        STATE['structure_centered'] = str(src)
        ok('Using the model as is (already centered).')
        if cmap:
            org, start, vox, n = _map_origin(cmap)
            if np.allclose(org, ((-n) // 2) * vox, atol=vox.max()):
                STATE['map_centered'] = cmap
    elif mode == 'map':
        org, start, vox, n = _map_origin(cmap)
        if abs(n[0] * vox[0] - box_A) > 0.01 * box_A:
            warn(f'The map box ({n[0] * vox[0]:.0f} Å) differs from the particle box ({box_A:.0f} Å); '
                 'centering on the map may not match the particles. Consider "Shift by half the particle box".')
        if np.any(org != 0) or np.any(start != 0):
            off = org if np.any(org != 0) else start * vox
            shift = ((-n) // 2) * vox - off
            fmt = lambda v: '(' + ', '.join(f'{x:.2f}' for x in v) + ')'
            warn(f'The map header has a non-zero origin {fmt(off)} Å; applying the equivalent shift {fmt(shift)} Å '
                 'directly instead of cryosphere_center_origin (which assumes origin 0).')
            dst = out_dir / f'{src.stem}_centered.pdb'
            shift_pdb(src, dst, shift)
            STATE['structure_centered'] = str(dst)
        else:
            shift = ((-n) // 2) * vox if (vox[0] == vox[1] == vox[2] and n[0] == n[1] == n[2]) else \
                np.array([(-n[0]) // 2 * vox[0], (-n[1]) // 2 * vox[1], (-n[2]) // 2 * vox[2]])
            run_cmd(['cryosphere_center_origin', '--pdb_file_path', str(src.resolve()), '--mrc_file_path', str(Path(cmap).resolve())],
                cwd=out_dir)
            STATE['structure_centered'] = str(out_dir / f'{src.stem}_centered.pdb')
            STATE['map_centered'] = str(out_dir / f'{Path(cmap).stem}_centered.mrc')
            if not Path(STATE['structure_centered']).exists():
                fail('cryosphere_center_origin did not write the centered model.')
    else:
        shift = np.full(3, ((-ds['box']) // 2) * ds['apix'])   # same convention as cryoSPHERE's EMAN2Grid
        dst = out_dir / f'{src.stem}_centered.pdb'
        shift_pdb(src, dst, shift)
        STATE['structure_centered'] = str(dst)
        ok(f'Shifted the model by {shift[0]:.2f} Å along x, y and z (half the particle box).')
    shift = np.asarray(shift, dtype=float)
    (out_dir / 'centering.json').write_text(json.dumps({
        'source_model': str(src), 'centered_model': STATE['structure_centered'], 'shift_A': [float(v) for v in shift],
        'note': 'centered = original + shift_A; to go back to the original map frame subtract shift_A'}, indent=1))
    prev_fsc = STATE.get('lp_fsc') or {}
    if STATE.get('centering_shift') != [float(v) for v in shift] or prev_fsc.get('for') != STATE['structure_centered']:
        STATE.pop('lp_fsc', None)                 # an FSC result belongs to the previous centering
    STATE['centering_shift'] = [float(v) for v in shift]
    new = load_polymer(STATE['structure_centered']).coord
    c = new.mean(0)
    outside = np.mean(np.any(np.abs(new) > box_A / 2, axis=1))
    ok(f'Centered model: {STATE["structure_centered"]}  (centroid now {np.linalg.norm(c):.1f} Å from the box center)')
    if outside > 0:
        warn(f'{outside:.1%} of the residues lie outside the particle box: check the fit and centering in 3.2.')
    if np.linalg.norm(c) > 0.25 * box_A:
        warn('The model is still far from the box center. Check that it was fitted into the consensus map of these particles.')
    save_state()

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

In [ ]:
#@title 3.2 · (Optional) Check the fit and pick a low-pass filter from the FSC { display-mode: "form" }
#@markdown Renders the model the way cryoSPHERE sees it (one Gaussian per residue) with `cryosphere_structure_to_volume`, overlays it on the consensus map and computes their Fourier shell correlation.
#@markdown The resolution where the FSC drops below the threshold is a good `lp_bandwidth` (frequencies the model cannot explain are filtered out of the images).
fsc_threshold = 0.5  #@param {type:"number"}
sigma_gmm = 2.0  #@param {type:"number"}

def _main():
    need('dataset')
    import mrcfile
    if not STATE.get('structure_centered'):
        fail('Run 3.1 first.')
    cmap = STATE.get('map_centered') or STATE.get('map_file')
    if not cmap:
        info('No consensus map: skipping the fit check. lp_bandwidth stays as set in 4.1.')
        return
    ds = STATE['dataset']
    work = Path(STATE['data_dir']) / 'fsc_check'
    work.mkdir(exist_ok=True)
    write_yaml(work / 'images.yaml', {'apix': ds['apix'], 'Npix': ds['box'], 'Npix_downsize': ds['box'], 'sigma_gmm': float(sigma_gmm)})
    gmm_path = work / 'model_gmm_volume.mrc'
    run_cmd(['cryosphere_structure_to_volume', '--image_yaml', str(work / 'images.yaml'),
         '--structure_path', STATE['structure_centered'], '--output_path', str(gmm_path)], cwd=work)
    with mrcfile.open(gmm_path, permissive=True) as m:
        gmm = m.data.astype(np.float32).copy()
    with mrcfile.open(cmap, permissive=True) as m:
        vol = m.data.astype(np.float32).copy()
        vox = float(m.voxel_size.x)
    if vol.ndim != 3 or len(set(vol.shape)) != 1:
        warn(f'The map is not a cube ({vol.shape}); skipping the FSC.')
        return
    if vol.shape[0] != ds['box'] or abs(vox - ds['apix']) > 0.01 * ds['apix']:
        if abs(vol.shape[0] * vox - ds['box'] * ds['apix']) > 0.01 * ds['box'] * ds['apix']:
            warn(f'The map box ({vol.shape[0] * vox:.0f} Å) differs from the particle box ({ds["box"] * ds["apix"]:.0f} Å); skipping the FSC.')
            return
        info(f'Resampling the map from {vol.shape[0]} px at {vox:.3f} Å to {ds["box"]} px at {ds["apix"]:.3f} Å.')
        vol = fourier_resize(vol, ds['box'])
    mask = model_mask(gmm)
    freq, fsc = fsc_curve(gmm * mask, vol * mask, ds['apix'])
    cross = fsc_crossing(freq, fsc, fsc_threshold)
    fig, axes = plt.subplots(1, 4, figsize=(17, 3.9), gridspec_kw={'width_ratios': [1.6, 1, 1, 1]})
    axes[0].plot(freq[1:], fsc[1:], color='#1f77b4', lw=2)
    axes[0].axhline(fsc_threshold, color='gray', ls='--', lw=1)
    axes[0].set_xlabel('spatial frequency (1/Å)'); axes[0].set_ylabel('FSC'); axes[0].set_ylim(-0.1, 1.02)
    axes[0].set_title('FSC: model (GMM) vs consensus map, inside a model mask')
    if cross:
        axes[0].axvline(cross, color='#d62728', lw=1)
        axes[0].text(cross, 0.9, f' {1 / cross:.1f} Å', color='#d62728')
    for ax, (name, axis) in zip(axes[1:], [('projection along Z', 0), ('projection along Y', 1), ('projection along X', 2)]):
        pm, pg = vol.sum(axis), gmm.sum(axis)
        ax.imshow(pm, cmap='gray', origin='lower')
        if pg.max() > 0:
            ax.contour(pg, levels=[0.25 * pg.max()], colors='#ff7f0e', linewidths=1.2, origin='lower')
        ax.set_title(name, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle('Consensus map (gray) and the model as cryoSPHERE renders it (orange outline): they should overlap', fontsize=11)
    fig.tight_layout()
    plt.show()
    low = fsc[1:max(2, int(len(fsc) * 0.15))].mean()
    nyquist = 2 * ds['apix']
    lp = float(round(1 / cross, 1)) if cross else None
    result = {'value': None, 'for': STATE['structure_centered']}
    if low < 0.5:
        warn(f'The FSC is low even at low resolution (mean {low:.2f}): the model is probably not fitted or not centered '
             'like the map. Check the overlay above; no low-pass filter is suggested until this is fixed.')
        result = None
    elif lp is None or lp <= nyquist * 1.05:
        ok(f'The FSC stays above {fsc_threshold} up to Nyquist ({nyquist:.1f} Å): no low-pass filter needed ("auto" in 4.1 uses none).')
    elif lp > 25:
        warn(f'FSC = {fsc_threshold} only at {lp} Å. That is unusually coarse and would remove most of the signal, so "auto" '
             'will not use it. Check the fit, or set lp_bandwidth in 4.1 yourself.')
        result = None
    else:
        result['value'] = lp
        ok(f'FSC = {fsc_threshold} at {lp} Å → suggested lp_bandwidth: {lp} Å (used by "auto" in 4.1).')
    if result is None:
        STATE.pop('lp_fsc', None)
    else:
        STATE['lp_fsc'] = result
    save_state()

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

---
## 4 · Configure the run

This step writes the two files cryoSPHERE reads, `parameters.yaml` and `images.yaml`, into a run folder (`<project>/runs/<run_name>`).

**Segmentation** is the key modelling choice: cryoSPHERE splits the protein into segments that move as rigid bodies, and learns where the segment boundaries are.
* *Whole protein*: one segmentation along the whole residue sequence, as if the protein were a single chain (protein chains in file order, then nucleic-acid chains, which is cryoSPHERE's residue order). Easiest. Keep at least ~50 residues per segment: with fewer, cryoSPHERE becomes numerically unstable (NaN losses). `N_segments = 0` picks about one segment per 50 residues, up to 20.
* *Custom parts*: segment only the regions you expect to move, e.g. `A:all:4, C:120-269:3` (chain:residues:segments); everything else stays fixed. Look at the chain table in 2.2 for chain names and residue numbers.
* *Auto*: uses the segmentation of a `parameters*.yaml` shipped with the data (as in the example dataset), otherwise *Whole protein*.

In [ ]:
#@title 4.1 · Configure the run { display-mode: "form" }
run_name = "run1"  #@param {type:"string"}
#@markdown ### Training
N_epochs = 50  #@param {type:"integer"}
batch_size = 128  #@param {type:"integer"}
latent_dimension = 8  #@param {type:"integer"}
#@markdown Train on a random subset of particles (0 = all), e.g. 2000 for a quick test:
max_particles = 0  #@param {type:"integer"}
#@markdown ### Segmentation
segmentation = "Auto"  #@param ["Auto", "Whole protein", "Custom parts", "From a parameters YAML in the data folder"]
#@markdown *Whole protein*: number of segments (0 = auto: about one per 50 residues, between 2 and 20)
N_segments = 0  #@param {type:"integer"}
#@markdown *Custom parts*: `CHAIN:FIRST-LAST:N_SEGMENTS`, comma-separated (`all` = whole chain)
custom_parts = "A:all:6"  #@param {type:"string"}
residue_numbering = "PDB residue numbers"  #@param ["PDB residue numbers", "0-based position within the chain"]
#@markdown ### Images
#@markdown Downsample images to this many pixels per side (0 = auto: the data's own setting, else no downsampling up to 256 px):
Npix_downsize = 0  #@param {type:"integer"}
#@markdown Low-pass filter in Å: `auto` (the FSC suggestion from 3.2 or the data's YAML, else none), `none`, or a number such as `15`:
lp_bandwidth = "auto"  #@param {type:"string"}
#@markdown Circular mask applied to the images before the encoder (1 = touches the box edges):
input_mask_radius = 1.0  #@param {type:"number"}
#@markdown ### Advanced (defaults work well)
learning_rate = 0.00003  #@param {type:"number"}
learning_rate_segmentation = 0.0003  #@param {type:"number"}
clashing_loss = "Auto"  #@param ["Auto", "Full", "Light (large proteins)"]
continuity_loss_weight = 0.1  #@param {type:"number"}
clashing_loss_weight = 0.1  #@param {type:"number"}
encoder_hidden_layers = "512, 256, 64, 64"  #@param {type:"string"}
decoder_hidden_layers = "512, 512"  #@param {type:"string"}
sigma_gmm = 2.0  #@param {type:"number"}
tau_segmentation = 0.05  #@param {type:"number"}
amortized = True  #@param {type:"boolean"}
seed = 123  #@param {type:"integer"}
#@markdown ### Checkpoints
#@markdown A model checkpoint is saved every epoch (often 50–150 MB each). Keep the most recent ones (0 = keep all) …
keep_last_checkpoints = 3  #@param {type:"integer"}
#@markdown … and also keep every N-th epoch for good (0 = none). With a Drive project these are the checkpoints copied to Drive during training.
keep_every_n_epochs = 10  #@param {type:"integer"}
#@markdown ### Continue a previous run
#@markdown Name of an earlier run to start from (its architecture and segmentation are reused), and the epoch (-1 = latest):
resume_from_run = ""  #@param {type:"string"}
resume_epoch = -1  #@param {type:"integer"}
overwrite_existing_run = False  #@param {type:"boolean"}

def _ints(text, label):
    try:
        vals = [int(v) for v in re.split(r'[,\s]+', text.strip()) if v]
    except ValueError:
        fail(f'{label}: use comma-separated integers, e.g. "512, 256".')
    if not vals or min(vals) < 1:
        fail(f'{label}: give at least one positive layer size.')
    return vals

def _data_yaml(data_dir, pattern):
    for p in rank_by_name(find_files(data_dir, ('.yaml', '.yml')), prefer=('two', 'parameters', 'images')):
        if re.search(pattern, p.name, re.I):
            try:
                y = lenient_yaml_load(p)
                if y:
                    return p, y
            except Exception:
                pass
    return None, {}

def _main():
    need('dataset')
    import torch
    ds = STATE['dataset']
    data_dir = Path(STATE['data_dir'])
    name = re.sub(r'[^\w.\-]+', '_', run_name.strip())
    if not name:
        fail('Give the run a name.')
    run_dir = runs_dir() / name
    notes = []
    rec = bg_record()
    if background_run_alive() and rec:
        if Path(rec['run_dir']) == run_dir:
            fail(f'Run "{name}" is training in the background right now. Choose another run_name, or stop it in 5.3 first.')
        if resume_from_run.strip() and Path(rec['run_dir']) == runs_dir() / resume_from_run.strip():
            fail(f'"{resume_from_run}" is still training in the background; continue from it once it has finished (or stop it in 5.3).')
    # ---------- resume (validated before anything is deleted)
    resume = {'model': None, 'segmentation': None}
    prev_params = prev_images = None
    if resume_from_run.strip():
        if name == resume_from_run.strip():
            fail('Use a new run_name when continuing a run (cryoSPHERE restarts epoch numbering and would overwrite checkpoints).')
        prev = runs_dir() / resume_from_run.strip()
        if not (prev / 'parameters.yaml').exists():
            fail(f'No run called "{resume_from_run}" in {runs_dir()}.')
        eps = list_checkpoints(prev)
        if not eps:
            fail(f'Run "{resume_from_run}" has no checkpoints.')
        ep = max(eps) if resume_epoch < 0 else resume_epoch
        if ep not in eps:
            fail(f'Epoch {ep} of "{resume_from_run}" is not available; kept epochs: {sorted(eps)}.')
        resume = {'model': str(eps[ep][0]), 'segmentation': str(eps[ep][1])}
        prev_params = yaml.safe_load((prev / 'parameters.yaml').read_text())
        prev_images = yaml.safe_load((prev / 'images.yaml').read_text())
        resumed_epoch = ep
        notes.append(f'continuing from {resume_from_run} epoch {ep}: base structure, segmentation, network sizes, '
                     'image size and low-pass filter are taken from that run. N_epochs counts additional epochs, and '
                     'the optimizer state starts fresh')
    if run_dir.exists() and list_checkpoints(run_dir):
        if not overwrite_existing_run:
            fail(f'Run "{name}" already has checkpoints. Pick another run_name, tick overwrite_existing_run, '
                 'or continue it in a new run with resume_from_run.')
        if any(str(run_dir) in str(p) for p in resume.values() if p):
            fail('Refusing to overwrite the run you are continuing from.')
    work_dir = WORK_ROOT / Path(STATE['project_dir']).name / name if on_drive(run_dir) else run_dir
    if work_dir != run_dir and keep_every_n_epochs < 1:
        fail('With a Google Drive project, checkpoints reach Drive every keep_every_n_epochs epochs (and at the end); '
             'set keep_every_n_epochs to 1 or more (e.g. 5) so a disconnect cannot lose the whole run.')
    if run_dir.exists() and overwrite_existing_run:
        shutil.rmtree(run_dir)
        if work_dir != run_dir:
            shutil.rmtree(work_dir, ignore_errors=True)
    if prev_params:
        structure = prev_params['base_structure_path']
        if not Path(structure).exists():
            fail(f'The base structure of "{resume_from_run}" ({structure}) is gone, probably because the Colab disk was reset. '
                 'Re-run 2.1, 2.2 and 3.1 with the same settings, then this cell.')
    else:
        structure = STATE.get('structure_centered')
        if structure and not Path(structure).exists():
            fail(f'The centered model {structure} is gone (Colab disk reset?). Re-run 2.1, 2.2 and 3.1.')
        if not structure:
            structure = STATE['structure_file']
    poly = load_polymer(structure)
    n_res = len(poly.chain_id)
    off = float(np.linalg.norm(poly.coord.mean(0)))
    if off > 0.25 * ds['box'] * ds['apix']:
        fail(f'The model is {off:.0f} Å from the box center, so it is not centered the way cryoSPHERE needs. Run 3.1 '
             '(again, if you re-ran 2.2), then this cell.')
    # ---------- images
    _, data_images = _data_yaml(data_dir, r'^images')
    _, data_params = _data_yaml(data_dir, r'param')
    if prev_params:
        nd = int(prev_images['Npix_downsize'])
    elif Npix_downsize > 0:
        nd = int(Npix_downsize)
    elif data_images.get('Npix') == ds['box'] and data_images.get('Npix_downsize'):
        nd = int(data_images['Npix_downsize'])
        notes.append(f'Npix_downsize = {nd} from the data folder\'s images.yaml')
    else:
        nd = min(ds['box'], 256)
    if nd > ds['box']:
        fail(f'Npix_downsize ({nd}) cannot exceed the image size ({ds["box"]}).')
    apix_down = ds['box'] * ds['apix'] / nd
    # ---------- low-pass filter
    lp_text = lp_bandwidth.strip().lower()
    if lp_text in ('', 'none', 'no', 'off'):
        lp = None
    elif lp_text == 'auto' and prev_params:
        lp = prev_params.get('lp_bandwidth')
    elif lp_text == 'auto':
        fsc_res = STATE.get('lp_fsc')
        if fsc_res is not None and fsc_res.get('for') == structure:
            lp = fsc_res.get('value')
            notes.append(f'lp_bandwidth = {lp} Å from the FSC in 3.2' if lp else 'no low-pass filter (the FSC in 3.2 did not call for one)')
        else:
            lp = data_params.get('lp_bandwidth') or None
            if lp:
                notes.append(f'lp_bandwidth = {lp} Å from the data folder\'s YAML')
    else:
        try:
            lp = float(lp_text)
        except ValueError:
            fail('lp_bandwidth must be "auto", "none" or a number in Å.')
    if lp is not None and lp <= 2 * apix_down:
        notes.append(f'lp_bandwidth {lp} Å is finer than the Nyquist limit ({2 * apix_down:.2f} Å), so it has no effect')
    # ---------- segmentation
    if prev_params:
        seg_cfg, how = prev_params['segmentation_config'], 'from the resumed run'
    else:
        mode = segmentation
        if mode == 'Auto':
            mode = 'Whole protein'
            if data_params.get('segmentation_config'):
                try:
                    seg_cfg, seg_notes = normalize_segmentation(data_params['segmentation_config'], poly)
                    mode, how = 'auto-yaml', 'from the data folder\'s parameters YAML'
                    notes += seg_notes
                except StepError as e:
                    warn(f'The segmentation in the data folder\'s YAML does not fit this structure ({e}); using the whole protein instead.')
        if mode == 'auto-yaml':
            pass
        elif mode == 'Whole protein':
            n_seg = int(N_segments) if N_segments > 0 else auto_n_segments(n_res)
            if not 1 <= n_seg <= n_res:
                fail(f'N_segments must be between 1 and the number of residues ({n_res}).')
            seg_cfg, how = {'part1': {'all_protein': True, 'N_segm': n_seg}}, 'whole protein' + (' (auto)' if N_segments <= 0 else '')
        elif mode == 'Custom parts':
            seg_cfg, how = parse_parts(custom_parts, poly, residue_numbering), 'custom parts'
        else:
            if not data_params.get('segmentation_config'):
                fail('No parameters YAML with a segmentation_config was found in the data folder.')
            seg_cfg, seg_notes = normalize_segmentation(data_params['segmentation_config'], poly)
            how = 'from the data folder\'s parameters YAML'
            notes += seg_notes
    for r in segmentation_risks(seg_cfg, poly):
        warn(f'Few residues per segment ({r}). cryoSPHERE becomes numerically unstable below ~{MIN_RES_PER_SEGMENT} residues '
             'per segment and the loss can turn NaN; consider fewer segments. Training stops automatically if that happens.')
    # ---------- particles subset
    n_total = ds['n']
    meta = STATE['metadata']
    run_dir.mkdir(parents=True, exist_ok=True)
    n_used = n_total
    if 0 < max_particles < n_total:
        if max_particles < 100:
            fail('max_particles must be at least 100.')
        sel = np.sort(np.random.default_rng(seed).choice(n_total, size=max_particles, replace=False))
        if STATE['kind'] == 'star':
            import starfile
            optics, parts = read_star(meta)
            sub = parts.iloc[sel].reset_index(drop=True)
            meta = str(run_dir / 'particles_subset.star')
            starfile.write({'optics': optics, 'particles': sub} if optics is not None else sub, meta, overwrite=True)
        else:
            meta = str(run_dir / 'particles_subset.cs')
            save_cs(read_cs(STATE['metadata'])[sel], meta)
        np.save(run_dir / 'particles_subset_indices.npy', sel)
        n_used = int(max_particles)
        notes.append(f'training on {n_used:,} of {n_total:,} particles (indices saved in particles_subset_indices.npy)')
    n_gpus = max(1, torch.cuda.device_count())
    if batch_size < 1 or batch_size > n_used // n_gpus:
        fail(f'batch_size must be between 1 and {n_used // n_gpus} (particles per GPU).')
    if N_epochs < 2:
        fail('N_epochs must be at least 2 (cryoSPHERE\'s linear KL schedule divides by N_epochs // 2).')
    latent = int(prev_params['latent_dimension']) if prev_params else int(latent_dimension)
    if latent < 2:
        fail('latent_dimension must be at least 2.')
    enc = prev_params['encoder']['hidden_dimensions'] if prev_params else _ints(encoder_hidden_layers, 'encoder_hidden_layers')
    dec = prev_params['decoder']['hidden_dimensions'] if prev_params else _ints(decoder_hidden_layers, 'decoder_hidden_layers')
    amort = bool(prev_params['amortized']) if prev_params else bool(amortized)
    # ---------- clashing loss memory estimate (all residue pairs × batch, float32, with gradients)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else 16e9
    full_mem = batch_size * n_res ** 2 * 4 * 4
    if clashing_loss == 'Auto':
        full = full_mem < 0.35 * gpu_mem
        if not full:
            notes.append(f'light clashing loss: the full one would need ~{human_size(full_mem)} of GPU memory for {n_res:,} residues')
    else:
        full = clashing_loss == 'Full'
        if full and full_mem > 0.6 * gpu_mem:
            warn(f'The full clashing loss needs ~{human_size(full_mem)} of GPU memory; expect out-of-memory errors. '
                 'Use the light version or a smaller batch_size.')
    params = {
        'name': name,
        'wandb': bool(STATE.get('wandb')),
        'wandb_project': STATE.get('wandb_project', 'cryoSPHERE'),
        'folder_path': str(work_dir),
        'base_structure_path': str(Path(structure).resolve()),
        'image_yaml': 'images.yaml',
        'particles_path': str(Path(STATE['particles_folder']).resolve()),
        'cs_star_file': {'file': str(Path(meta).resolve()), 'abinit': ds['abinit'], 'hetrefine': ds['hetrefine']},
        'segmentation_config': seg_cfg,
        'lp_bandwidth': float(lp) if lp else None,
        'loss_mask_radius': 1,
        'input_mask_radius': float(input_mask_radius),
        'latent_dimension': latent,
        'tau_segmentation': float(tau_segmentation),
        'device': 'GPU',
        'num_workers': int(min(4, os.cpu_count() or 2)),
        'amortized': amort,
        'N_images': int(n_used),
        'N_epochs': int(N_epochs),
        'batch_size': int(batch_size),
        'epsilon_kl': '1e-10',          # cryoSPHERE eval()s this value, so it must stay a string
        'seed': int(seed),
        'deterministic_cuda': False,
        'encoder': {'hidden_dimensions': [int(v) for v in enc]},
        'decoder': {'hidden_dimensions': [int(v) for v in dec]},
        'optimizer': {'name': 'adam', 'learning_rate': float(learning_rate), 'learning_rate_segmentation': float(learning_rate_segmentation)},
        'loss': {
            'clashing_loss': {'full_clashing_loss': bool(full), 'clashing_cutoff': 4, 'min_clashing_cutoff_pairs': 4,
                              'max_clashing_cutoff_pairs': 10, 'schedule': 'constant', 'beta': float(clashing_loss_weight)},
            'continuity_loss': {'schedule': 'constant', 'beta': float(continuity_loss_weight)},
            'KL_prior_latent': {'schedule': 'linear'},
            'KL_prior_segmentation_mean': {'schedule': 'linear'},
            'KL_prior_segmentation_std': {'schedule': 'linear'},
            'KL_prior_segmentation_proportions': {'schedule': 'cyclical', 'M': 5, 'R': 0.5},
            'l2_pen': {'schedule': 'constant', 'beta': 0.000001},
        },
        'resume_training': resume,
    }
    sigma = float(prev_images['sigma_gmm']) if prev_images else float(sigma_gmm)
    if prev_params:
        # cryoSPHERE restarts its epoch counter, so annealed KL weights would warm up from 0 again
        from cryosphere.model.loss import compute_beta_schedule
        prev_n = int(prev_params['N_epochs'])
        for term in ('KL_prior_latent', 'KL_prior_segmentation_mean', 'KL_prior_segmentation_std'):
            cfg = prev_params['loss'].get(term, {'schedule': 'linear'})
            if cfg.get('schedule') == 'cyclical':
                continue
            reached = float(compute_beta_schedule(resumed_epoch + 1, prev_n, cfg))
            if reached >= 1:
                params['loss'][term] = {'schedule': 'constant', 'beta': 1.0}
            else:
                remaining = max(1, round((1 - reached) * cfg.get('M', prev_n // 2)))
                params['loss'][term] = {'schedule': 'linear', 'M': int(remaining)}
                notes.append(f'{term}: the earlier run was still warming up this weight (β = {reached:.2f}); '
                             f'the warm-up restarts and reaches 1 after {remaining} epochs')
    images = {'apix': float(ds['apix']), 'Npix': int(ds['box']), 'Npix_downsize': int(nd), 'sigma_gmm': sigma}
    if Path(structure).resolve() != (run_dir / 'base_structure_centered.pdb').resolve():
        shutil.copyfile(structure, run_dir / 'base_structure_centered.pdb')
    for cj in (Path(structure).parent / 'centering.json', Path(structure).parent / 'centering_from_run.json'):
        if cj.exists():
            shutil.copyfile(cj, run_dir / 'centering.json')
            break
    params['base_structure_path'] = str((run_dir / 'base_structure_centered.pdb').resolve())
    write_yaml(run_dir / 'parameters.yaml', params)
    write_yaml(run_dir / 'images.yaml', images)
    if work_dir != run_dir:
        work_dir.mkdir(parents=True, exist_ok=True)
        write_yaml(work_dir / 'images.yaml', images)        # cryoSPHERE reads images.yaml from folder_path
    reread = yaml.safe_load((run_dir / 'parameters.yaml').read_text())
    assert isinstance(reread['epsilon_kl'], str) and reread['segmentation_config'] == seg_cfg
    ckpt_mb = (nd * nd * enc[0] + sum(a * b for a, b in zip(enc, enc[1:])) + latent * dec[0]) * 4 / 1e6
    n_seg = sum(p['N_segm'] for p in seg_cfg.values())
    print(f'Run folder      {run_dir}')
    print(f'Particles       {n_used:,} · {nd}×{nd} px at {apix_down:.2f} Å/px (from {ds["box"]} px)')
    print(f'Segmentation    {how}: {n_seg} segments in {len(seg_cfg)} part(s) over {n_res:,} residues')
    print(f'Training        {N_epochs} epochs · batch {batch_size} · latent dim {latent} · {"full" if full else "light"} clashing loss · lp_bandwidth {lp or "none"}')
    if work_dir != run_dir:
        print(f'Work folder     {work_dir} (local disk). Copied to Drive: logs and segmentations every epoch, a checkpoint '
              f'every {keep_every_n_epochs} epochs and the final one; a disconnect loses at most {keep_every_n_epochs - 1} epochs.')
    print(f'Checkpoints     ~{ckpt_mb:.0f} MB each; keeping the last {keep_last_checkpoints or "all"}'
          + (f' plus one every {keep_every_n_epochs} epochs' if keep_last_checkpoints and keep_every_n_epochs else ''))
    for n_ in notes:
        info(n_)
    display(HTML('<details><summary><b>parameters.yaml</b> (click to expand)</summary><pre>'
                 + (run_dir / 'parameters.yaml').read_text() + '</pre></details>'
                 + '<details><summary><b>images.yaml</b></summary><pre>' + (run_dir / 'images.yaml').read_text() + '</pre></details>'))
    STATE['run_dir'] = str(run_dir)
    STATE.setdefault('runs', {})[name] = {'run_dir': str(run_dir), 'N_epochs': int(N_epochs),
                                          'keep_last': int(keep_last_checkpoints), 'keep_every': int(keep_every_n_epochs)}
    save_state()
    ok(f'Configuration written. Next: 5.1 to train run "{name}".')

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

---
## 5 · Train

Training runs the `cryosphere_train` command on the files written in 4.1. Each epoch goes through every particle once; at the end of each epoch cryoSPHERE saves the model (`ckpt<epoch>.pt`) and the segmentation (`seg<epoch>.pt`) in `<run>/cryoSPHERE/`.

* **Foreground** shows live progress and the loss curves in the cell. Stopping the cell (■) stops training and keeps all checkpoints.
* **Background** returns immediately so you can use the rest of the notebook; follow it with 5.2. Keep this browser tab open, because Colab may recycle idle runtimes.

Colab sessions are time-limited (around 12 h on the free tier, longer with Pro). With project storage on Google Drive, you can continue an interrupted run later with *Continue a previous run* in 4.1.

In [ ]:
#@title 5.1 · Train cryoSPHERE { display-mode: "form" }
training_mode = "Foreground: live progress in this cell"  #@param ["Foreground: live progress in this cell", "Background: keep using the notebook"]

# Companion process for background runs: prunes old checkpoints in the local work folder, copies the kept ones to a
# Drive project folder, and stops training if the loss turns NaN. Standard library only.
_WATCHER = """
import os, re, sys, time, shutil, signal
from pathlib import Path
run, work, pid, keep_last, keep_every = Path(sys.argv[1]), Path(sys.argv[2]), int(sys.argv[3]), int(sys.argv[4]), int(sys.argv[5])
src, dst = work / 'cryoSPHERE', run / 'cryoSPHERE'
def alive(p):
    try:
        os.kill(p, 0)
        with open(f'/proc/{p}/stat') as f:
            return f.read().rsplit(')', 1)[1].split()[0] != 'Z'
    except OSError:
        return False
def epochs(kind):
    return {int(m.group(1)): f for f in src.glob(kind + '*.pt') if (m := re.fullmatch(kind + r'(\\d+)\\.pt', f.name))}
def copy(f):
    t = dst / f.name
    if t.exists() and t.stat().st_size == f.stat().st_size:
        return
    shutil.copyfile(f, str(t) + '.partial')
    os.replace(str(t) + '.partial', t)
while True:
    running = alive(pid)
    ck, sg = epochs('ckpt'), epochs('seg')
    eps = sorted(e for e in ck if e in sg)
    if work != run and src.exists():
        dst.mkdir(parents=True, exist_ok=True)
        keep = {e for e in eps if keep_every > 0 and (e + 1) % keep_every == 0} | ({eps[-1]} if eps and not running else set())
        for f in [sg[e] for e in eps] + [ck[e] for e in keep] + list(src.glob('*.log')):
            try:
                copy(f)
            except OSError:
                pass
    if keep_last > 0 and not str(src).startswith('/content/drive'):
        for e in eps[:-keep_last]:
            if not (keep_every > 0 and (e + 1) % keep_every == 0):
                try:
                    ck[e].unlink()
                except OSError:
                    pass
    log = src / 'run.log'
    if running and log.exists() and re.search(r'Correlation loss: -?nan', log.read_text(errors='replace')):
        (run / '.nan_abort').write_text('Correlation loss became NaN')
        try:
            os.killpg(os.getpgid(pid), signal.SIGINT)
            time.sleep(10)
            os.killpg(os.getpgid(pid), signal.SIGKILL)
        except OSError:
            pass
    if not running:
        break
    time.sleep(20)
"""

_HINTS = [(r'CUDA out of memory', 'GPU out of memory: lower batch_size, set Npix_downsize (e.g. 128) or use the light clashing loss in 4.1.'),
          (r'Address already in use|EADDRINUSE', 'Port 12355 is busy: another cryoSPHERE process is running. Stop it in 5.3, or use Runtime → Disconnect and delete runtime.'),
          (r'API key', 'W&B login failed: re-run 1.5, or set wandb_mode to Off in 1.5 and re-run 4.1 (the choice is stored in parameters.yaml).'),
          (r'duplicate residue', 'The model has repeated residue numbers in a chain: re-run 2.2 (it renumbers them), 3.1 and 4.1.'),
          (r'No such file or directory', 'A file referenced in parameters.yaml is missing; re-run 2.1, 2.2, 3.1 and 4.1 (the Colab disk may have been reset).'),
          (r'\bKilled\b|SIGKILL', 'The process was killed, usually because the runtime ran out of RAM: lower batch_size.'),
          (r'\bnan\b', 'Losses became NaN: see the advice on segments in 4.1, and check the CTF columns of your metadata.')]

def _main():
    global _BG_POPEN
    need('run_dir')
    run_dir = Path(STATE['run_dir'])
    info_ = STATE['runs'][run_dir.name]
    params_path = run_dir / 'parameters.yaml'
    params = yaml.safe_load(params_path.read_text())
    work = Path(params['folder_path'])
    needed = [params['base_structure_path'], params['cs_star_file']['file'], params['particles_path']]
    for f in needed:
        if not Path(f).exists():
            fail(f'{f} is missing (the Colab disk was probably reset). Re-run 2.1, 2.2 and 3.1 with the same settings, then 4.1.')
    for f in (params['resume_training']['model'], params['resume_training']['segmentation']):
        if f and not Path(f).exists():
            fail(f'The checkpoint to continue from ({f}) no longer exists (it may have been pruned). Re-run 4.1 to pick a kept epoch.')
    if list_checkpoints(run_dir):
        fail(f'Run "{run_dir.name}" already has checkpoints. To train more, use resume_from_run in 4.1 with a new run_name.')
    ensure_gpu_free_for_cryosphere('training')
    work.mkdir(parents=True, exist_ok=True)
    (run_dir / '.nan_abort').unlink(missing_ok=True)
    env = {'WANDB_DIR': str(work), 'WANDB_SILENT': 'true'}
    if params['wandb']:
        if not os.environ.get('WANDB_API_KEY'):
            fail('W&B is on but no API key is loaded in this session: re-run 1.5 (or switch it Off and re-run 4.1).')
    else:
        env['WANDB_MODE'] = 'disabled'
    if work != run_dir:
        info(f'The project is on Google Drive: checkpoints are written to the local disk ({work}); logs, segmentations, '
             f'a checkpoint every {info_["keep_every"]} epochs and the final one are copied to Drive.')
    cmd = ['cryosphere_train', '--experiment_yaml', str(params_path)]
    log = run_dir / 'train_stdout.log'
    if training_mode.startswith('Background'):
        print('$ ' + ' '.join(cmd))
        with open(log, 'ab') as f:
            _BG_POPEN = subprocess.Popen(cmd, cwd=work, env={**os.environ, 'PYTHONUNBUFFERED': '1', **env}, stdout=f,
                                         stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)
        watcher = WORK_ROOT / f'.watch_{_BG_POPEN.pid}.py'
        watcher.parent.mkdir(parents=True, exist_ok=True)
        watcher.write_text(_WATCHER)
        subprocess.Popen([sys.executable, str(watcher), str(run_dir), str(work), str(_BG_POPEN.pid),
                          str(info_['keep_last']), str(info_['keep_every'])], start_new_session=True,
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        set_bg_record({'pid': _BG_POPEN.pid, 'run_dir': str(run_dir), 'project_dir': STATE['project_dir'],
                       'keep_every': info_['keep_every'], 'started': time.strftime('%Y-%m-%d %H:%M:%S')})
        ok(f'Training started in the background (pid {_BG_POPEN.pid}). Follow it with 5.2; stop it with 5.3. '
           'It also stops by itself if the loss turns NaN.')
        return
    monitor = TrainingMonitor(run_dir, params['N_epochs'], info_['keep_last'], info_['keep_every'])
    try:
        rc, tail = stream_process(cmd, log, cwd=work, env=env, monitor=monitor, echo_filter=quiet_filter)
    finally:
        sync_run(run_dir, info_['keep_every'], final=True)
    if rc != 0:
        text = '\n'.join(tail)
        print(text[-4000:])
        hints = [h for k, h in _HINTS if re.search(k, text)]
        fail(f'cryosphere_train stopped with exit code {rc}. ' + (' '.join(hints) or f'Full log: {log}'))
    nan_ep = first_nan_epoch(parse_run_logs(run_dir))
    if nan_ep is not None:
        fail(f'Epoch {nan_ep}: {NAN_HELP}')
    ok(f'Training finished: {params["N_epochs"]} epochs. Kept checkpoints for epochs {sorted(list_checkpoints(run_dir))}.')

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

In [ ]:
#@title 5.2 · Monitor training: loss curves, checkpoints, GPU { display-mode: "form" }
#@markdown Leave `run_to_monitor` empty to follow the background run (or else the current run). With `keep_refreshing`, the cell updates every minute until the background run ends.
run_to_monitor = ""  #@param {type:"string"}
keep_refreshing = False  #@param {type:"boolean"}

def _snapshot(run_dir, handle_text, handle_plot):
    rec = bg_record()
    alive = background_run_alive() and rec and rec['run_dir'] == str(run_dir)
    log = run_dir / 'train_stdout.log'
    progress, lines = '', []
    if log.exists():
        raw = log.read_bytes()[-200_000:].decode('utf-8', 'replace')
        pieces = [p for p in re.split(r'[\r\n]', raw) if p.strip()]
        prog = [p for p in pieces if TQDM_RE.search(p)]
        progress = prog[-1].strip() if prog else ''
        lines = [p for p in pieces if not TQDM_RE.search(p) and quiet_filter(p)][-8:]
    eps = list_checkpoints(run_dir)
    segs = seg_epochs(run_dir)
    params = yaml.safe_load((run_dir / 'parameters.yaml').read_text())
    gpu = subprocess.run('nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv,noheader',
                         shell=True, capture_output=True, text=True).stdout.strip()
    done = (max(segs) + 1) if segs else 0
    status = 'running in the background' if alive else ('finished' if done >= params['N_epochs'] else 'not running')
    m = TQDM_RE.search(progress)
    html = (f'<b>{run_dir.name}</b>: {status} · {done}/{params["N_epochs"]} epochs done'
            + (f' · current epoch: batch {m.group(2)}/{m.group(3)} [{m.group(4)}]' if m and alive else '')
            + f'<br>Checkpoints available: {sorted(eps) if eps else "none yet"}'
            + (f'<br>GPU: {gpu}' if gpu else '')
            + ('<br><small>Recent output:</small><pre style="font-size:11px">' + '\n'.join(lines) + '</pre>' if lines else ''))
    df = parse_run_logs(run_dir)
    nan_ep = first_nan_epoch(df)
    if (run_dir / '.nan_abort').exists():
        html += f'<br><b style="color:#d93025">⚠️ Training was stopped automatically because the loss became NaN. {NAN_HELP}</b>'
    elif nan_ep is not None:
        html += f'<br><b style="color:#d93025">⚠️ Loss is NaN since epoch {nan_ep}. {NAN_HELP}</b>'
    handle_text.update(HTML(html))
    fig = plot_losses(df, f'{run_dir.name}: training curves (first batch of each epoch)')
    if fig is not None:
        handle_plot.update(fig)
        plt.close(fig)
    return alive

def _main():
    rec = bg_record()
    if run_to_monitor.strip():
        run_dir = runs_dir() / run_to_monitor.strip()
    elif background_run_alive():
        run_dir = Path(rec['run_dir'])
        info(f'Following the background run {run_dir.name}.')
    else:
        run_dir = Path(STATE['run_dir']) if STATE.get('run_dir') else None
    if run_dir is None or not (run_dir / 'parameters.yaml').exists():
        fail('No run to monitor: configure one in 4.1 (or type an existing run name).')
    handle_text = display(HTML(''), display_id=True)
    handle_plot = display(HTML('<i>No epoch finished yet.</i>'), display_id=True)
    alive = _snapshot(run_dir, handle_text, handle_plot)
    try:
        while keep_refreshing and alive:
            time.sleep(60)
            alive = _snapshot(run_dir, handle_text, handle_plot)
    except KeyboardInterrupt:
        info('Stopped refreshing (training continues in the background).')
    others = [p.name for p in run_dir.parent.iterdir() if (p / 'parameters.yaml').exists()]
    if len(others) > 1:
        info(f'Runs in this project: {", ".join(sorted(others))}')

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

In [ ]:
#@title 5.3 · Stop a background training run { display-mode: "form" }
#@markdown Checkpoints written so far are kept; continue later with *Continue a previous run* in 4.1.
confirm_stop = False  #@param {type:"boolean"}

def _main():
    rec = bg_record()
    if background_run_alive():
        targets, label = [rec['pid']], f'the background run {Path(rec["run_dir"]).name} (pid {rec["pid"]})'
    else:
        procs = cryosphere_processes()
        if not procs:
            set_bg_record(None)
            info('No cryoSPHERE process is running.')
            return
        targets, label = [p.pid for p in procs], 'cryoSPHERE process(es) not started by 5.1 here: ' + '; '.join(describe_process(p) for p in procs)
    if not confirm_stop:
        info(f'Running: {label}. Tick confirm_stop and run this cell again to stop it.')
        return
    for pid in targets:
        kill_process_group(pid)
    if globals().get('_BG_POPEN') is not None:
        _BG_POPEN.poll()
    if rec and Path(rec['run_dir']).exists():
        sync_run(rec['run_dir'], rec.get('keep_every', 0), final=True)
        ok(f'Stopped. Checkpoints available for {Path(rec["run_dir"]).name}: {sorted(list_checkpoints(rec["run_dir"]))}')
    else:
        ok(f'Stopped {label}.')
    set_bg_record(None)

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

---
## 6 · Analyze

6.1 runs `cryosphere_analyze` to encode every particle into the latent space (`z.npy`) and to build structures along the main principal components; the other steps work from its output. They all use a single checkpoint, by default the last one.

Results land in `<run>/analysis/epoch_<N>/`. The predicted structures contain one C-alpha per residue (that is what cryoSPHERE models), which you can open directly in ChimeraX or PyMOL.

In [ ]:
#@title 6.1 · Encode all particles and run a PCA of the latent space { display-mode: "form" }
run_to_analyze = ""  #@param {type:"string"}
#@markdown Epoch to analyze (-1 = latest kept checkpoint):
epoch = -1  #@param {type:"integer"}
n_principal_components = 3  #@param {type:"slider", min:1, max:6, step:1}
structures_per_component = 10  #@param {type:"integer"}
#@markdown Use one particle in N for the PCA (speeds up very large datasets):
thinning = 1  #@param {type:"integer"}
reuse_existing_latents = True  #@param {type:"boolean"}

def _main():
    run_dir = runs_dir() / run_to_analyze.strip() if run_to_analyze.strip() else (Path(STATE['run_dir']) if STATE.get('run_dir') else None)
    if run_dir is None or not (run_dir / 'parameters.yaml').exists():
        fail('No run found: train one first (4.1 and 5.1), or type the name of an existing run.')
    ensure_gpu_free_for_cryosphere('the analysis')
    eps = list_checkpoints(run_dir)
    if not eps:
        fail(f'Run "{run_dir.name}" has no checkpoints yet.')
    ep = max(eps) if epoch < 0 else epoch
    if ep not in eps:
        fail(f'Epoch {ep} is not available; kept epochs: {sorted(eps)}.')
    ckpt, seg = eps[ep]
    cfg_path, params = analysis_config(run_dir)
    latent = int(params['latent_dimension'])
    n_pc = min(int(n_principal_components), latent - 1)
    if n_pc < n_principal_components:
        info(f'Using {n_pc} principal components (the latent space has {latent} dimensions).')
    out = run_dir / 'analysis' / f'epoch_{ep}'
    out.mkdir(parents=True, exist_ok=True)
    for old in list(out.glob('pc*/structure_z_*.*')) + list(out.glob('pc*/pca.png')):
        old.unlink()                     # traversals from an earlier run of this cell would mix into the new ones
    cmd = ['cryosphere_analyze', '--experiment_yaml', cfg_path, '--model', ckpt, '--segmenter', seg, '--output_path', out,
           '--dimensions', *range(n_pc), '--num_points', max(2, int(structures_per_component)),
           '--thinning', max(1, int(thinning)), '--no-generate_structures']
    if reuse_existing_latents and (out / 'z.npy').exists():
        cmd += ['--z', out / 'z.npy']
        info('Reusing the latent variables already computed for this epoch.')
    rc, tail = stream_process(cmd, out / 'analyze_stdout.log', cwd=out, env={'WANDB_MODE': 'disabled'},
                              monitor=ProgressLine('Encoding particles'), echo_filter=quiet_filter)
    if rc != 0:
        print('\n'.join(tail)[-4000:])
        fail(f'cryosphere_analyze failed (exit code {rc}); full log in {out / "analyze_stdout.log"}.')
    z = np.load(out / 'z.npy')
    STATE.update({'analysis_dir': str(out), 'analysis_run': str(run_dir), 'analysis_epoch': ep, 'analysis_npc': n_pc})
    save_state()
    ok(f'{len(z):,} particles encoded into a {z.shape[1]}-dimensional latent space → {out / "z.npy"}')
    from sklearn.decomposition import PCA
    pca = PCA().fit(z[::max(1, int(thinning))])
    zp = pca.transform(z)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
    axes[0].bar(np.arange(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_, color='#1f77b4')
    axes[0].set_xlabel('principal component'); axes[0].set_ylabel('fraction of variance'); axes[0].set_title('Explained variance')
    for ax, (a, b) in zip(axes[1:], [(0, 1), (0, 2) if z.shape[1] > 2 else (0, 1)]):
        hb = ax.hexbin(zp[:, a], zp[:, b], gridsize=60, bins='log', cmap='viridis', mincnt=1)
        ax.set_xlabel(f'PC{a + 1}'); ax.set_ylabel(f'PC{b + 1}'); ax.set_title(f'Particles in PC{a + 1}/PC{b + 1}')
        fig.colorbar(hb, ax=ax, label='particles (log scale)')
    fig.suptitle(f'{run_dir.name}, epoch {ep}: latent space', fontsize=12)
    fig.tight_layout()
    plt.show()
    for k in range(n_pc):
        png = out / f'pc{k}' / 'pca.png'
        if png.exists():
            print(f'PC{k + 1}: traversal points (red) used to build {out / f"pc{k}"}/structure_z_*.pdb')
            display(IPyImage(filename=str(png), width=520))

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

In [ ]:
#@title 6.2 · Watch the motion along a principal component (3D movie) { display-mode: "form" }
#@markdown Animates the structures generated along one principal component in 6.1, from the 5th to the 95th percentile of the particles.
principal_component = 1  #@param {type:"integer"}
color_by = "Segment"  #@param ["Segment", "Chain", "Displacement along the component"]
style = "Cartoon trace"  #@param ["Cartoon trace", "Spheres"]

def _main():
    need('analysis_dir')
    out, run_dir, ep = Path(STATE['analysis_dir']), Path(STATE['analysis_run']), STATE['analysis_epoch']
    k = int(principal_component) - 1
    files = sorted((out / f'pc{k}').glob('structure_z_*.pdb'), key=lambda p: int(re.findall(r'\d+', p.stem)[-1]))
    if not files:
        fail(f'No structures for PC{principal_component}: 6.1 built PC1–PC{STATE.get("analysis_npc", "?")}.')
    polys = [load_polymer(f) for f in files]
    coords = [p.coord for p in polys]
    base = polys[0]
    disp = np.max([np.linalg.norm(c - coords[0], axis=1) for c in coords], axis=0)
    _, labels = run_segmentation(run_dir, run_dir / 'cryoSPHERE' / f'seg{ep}.pt', base)
    sid = segment_ids(labels, len(base.chain_id))
    if color_by == 'Segment':
        colors, bf = [seg_color(i) for i in sid], None
    elif color_by == 'Chain':
        chains = list(pd.unique(base.chain_id))
        colors, bf = [SEG_COLORS[chains.index(c) % len(SEG_COLORS)] for c in base.chain_id], None
    else:
        colors, bf = None, disp
    print(f'PC{principal_component}: {len(files)} structures · largest residue displacement {disp.max():.1f} Å '
          f'(median {np.median(disp):.1f} Å)' + (' · red = moves most' if bf is not None else ''))
    view3d(coords, colors, base.chain_id, base.res_id, base.res_name, style='cartoon' if style.startswith('Cartoon') else 'sphere',
           bfactors=bf).show()
    fig, ax = plt.subplots(figsize=(15, 3))
    x = np.arange(len(disp))
    ax.bar(x, disp, color=[seg_color(i) for i in sid], width=1.0)
    for b in np.where(base.chain_id[1:] != base.chain_id[:-1])[0]:
        ax.axvline(b + 0.5, color='k', lw=0.8, ls=':')
    ax.set_xlim(-0.5, len(disp) - 0.5)
    ax.set_xlabel('residue (cryoSPHERE order: protein chains, then nucleic acids; dotted lines = chain breaks)'); ax.set_ylabel('displacement (Å)')
    ax.set_title(f'Per-residue displacement along PC{principal_component}, colored by segment (gray = not segmented)')
    fig.tight_layout()
    plt.show()

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

In [ ]:
#@title 6.3 · Inspect the learned segmentation { display-mode: "form" }
#@markdown Which residues move together, on the base structure, and how the segmentation evolved during training.
show_evolution = True  #@param {type:"boolean"}

def _ranges(chain_ids, res_ids):
    out, start, prev = [], None, None
    for c, r in zip(chain_ids, res_ids):
        if start is None or c != start[0] or r != prev[1] + 1:
            if start is not None:
                out.append(f'{start[0]}:{start[1]}-{prev[1]}' if start[1] != prev[1] else f'{start[0]}:{start[1]}')
            start = (c, r)
        prev = (c, r)
    if start is not None:
        out.append(f'{start[0]}:{start[1]}-{prev[1]}' if start[1] != prev[1] else f'{start[0]}:{start[1]}')
    return ', '.join(out)

def _main():
    import torch
    run_dir = Path(STATE.get('analysis_run') or STATE.get('run_dir') or '')
    if not (run_dir / 'parameters.yaml').exists():
        fail('Train a run first (4.1 and 5.1).')
    params = yaml.safe_load((run_dir / 'parameters.yaml').read_text())
    poly = load_polymer(params['base_structure_path'])
    seg_files = {}
    for d in (run_dir / 'cryoSPHERE', run_work_dir(run_dir) / 'cryoSPHERE'):
        seg_files.update({int(m.group(1)): p for p in d.glob('seg*.pt') if (m := re.fullmatch(r'seg(\d+)\.pt', p.name))})
    if not seg_files:
        fail('No segmentation saved yet (one is written at the end of every epoch).')
    ep = STATE.get('analysis_epoch') if STATE.get('analysis_run') == str(run_dir) and STATE.get('analysis_epoch') in seg_files else max(seg_files)
    labels = hard_segmentation(torch.load(seg_files[ep], map_location='cpu'), params['segmentation_config'], poly)
    sid = segment_ids(labels, len(poly.chain_id))
    rows = []
    for s in sorted(set(sid) - {-1}):
        sel = np.where(sid == s)[0]
        rows.append({'segment': s, 'color': seg_color(s), 'residues': len(sel), 'ranges (chain:residue#)': _ranges(poly.chain_id[sel], poly.res_id[sel])})
    print(f'{run_dir.name}, epoch {ep}: {len(rows)} non-empty segments' + (f' · {np.sum(sid < 0)} residues not segmented (fixed)' if np.any(sid < 0) else ''))
    df = pd.DataFrame(rows)
    display(df.style.hide(axis='index').map(lambda c: f'background-color: {c}; color: {c}', subset=['color']))
    view3d([poly.coord], [seg_color(i) for i in sid], poly.chain_id, poly.res_id, poly.res_name, animate=False, height=460).show()
    if show_evolution and len(seg_files) > 1:
        eps = sorted(seg_files)
        eps = eps[:: max(1, len(eps) // 80)]
        mat = np.array([segment_ids(hard_segmentation(torch.load(seg_files[e], map_location='cpu'), params['segmentation_config'], poly),
                                    len(poly.chain_id)) for e in eps], dtype=float)
        mat[mat < 0] = np.nan
        cmap = matplotlib.colors.ListedColormap([seg_color(i) for i in range(int(np.nanmax(mat)) + 1)])
        fig, ax = plt.subplots(figsize=(15, 3.5))
        ax.imshow(mat, aspect='auto', interpolation='nearest', cmap=cmap, vmin=-0.5, vmax=np.nanmax(mat) + 0.5)
        ax.set_yticks(range(len(eps))[:: max(1, len(eps) // 10)]); ax.set_yticklabels(eps[:: max(1, len(eps) // 10)])
        ax.set_xlabel('residue (cryoSPHERE order)'); ax.set_ylabel('epoch')
        ax.set_title('Segment of each residue during training (white = not segmented)')
        fig.tight_layout()
        plt.show()

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

In [ ]:
#@title 6.4 · Structures for chosen particles { display-mode: "form" }
#@markdown Particle numbers are 0-based rows of the metadata used for training (e.g. `0, 12, 100-120`), or `random:20`, or `all` (one PDB per particle; can take a while).
#@markdown For runs on a `max_particles` subset these are rows of the subset; `particles.csv` also lists the row in your original star/cs file.
particles = "random:10"  #@param {type:"string"}

def _parse_selection(text, n):
    t = text.strip().lower()
    if t == 'all':
        return np.arange(n)
    if t.startswith('random'):
        k = int(t.split(':')[1]) if ':' in t else 10
        return np.sort(np.random.default_rng(0).choice(n, size=min(k, n), replace=False))
    idx = []
    for part in re.split(r'[,\s]+', t):
        if not part:
            continue
        if '-' in part:
            a, b = part.split('-', 1)
            idx += list(range(int(a), int(b) + 1))
        else:
            idx.append(int(part))
    idx = np.array(sorted(set(idx)))
    if len(idx) == 0 or idx.min() < 0 or idx.max() >= n:
        fail(f'Particle numbers must be between 0 and {n - 1}.')
    return idx

def _main():
    need('analysis_dir')
    out = Path(STATE['analysis_dir'])
    z = np.load(out / 'z.npy')
    idx = _parse_selection(particles, len(z))
    if len(idx) > 5000:
        warn(f'Generating {len(idx):,} structures; this can take a long time and a lot of disk.')
    tag = 'all' if particles.strip().lower() == 'all' else f'{len(idx)}_particles'
    dest = out / f'particles_{tag}'
    paths = generate_from_latents(z[idx], dest, [f'particle_{i:07d}' for i in idx], 'structure generation')
    sub = Path(STATE['analysis_run']) / 'particles_subset_indices.npy'
    rows = np.load(sub)[idx] if sub.exists() else idx
    pd.DataFrame({'particle': idx, 'row_in_original_metadata': rows, 'pdb': [p.name for p in paths]}).to_csv(dest / 'particles.csv', index=False)
    ok(f'{len(paths)} structures in {dest}')
    show = paths[:12]
    polys = [load_polymer(p) for p in show]
    base = polys[0]
    print(f'Cycling through {len(show)} of them (particles {", ".join(str(i) for i in idx[:12])}); colored by chain.')
    chains = list(pd.unique(base.chain_id))
    view3d([p.coord for p in polys], [SEG_COLORS[chains.index(c) % len(SEG_COLORS)] for c in base.chain_id],
           base.chain_id, base.res_id, base.res_name).show()

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

In [ ]:
#@title 6.5 · Cluster the latent space and build one structure per cluster { display-mode: "form" }
#@markdown k-means on the latent variables of all particles; each cluster center is decoded into a representative structure.
n_clusters = 6  #@param {type:"slider", min:2, max:20, step:1}

def _main():
    need('analysis_dir')
    from sklearn.cluster import KMeans
    from sklearn.decomposition import PCA
    out = Path(STATE['analysis_dir'])
    z = np.load(out / 'z.npy')
    km = KMeans(n_clusters=int(n_clusters), n_init=10, random_state=0).fit(z)
    counts = np.bincount(km.labels_, minlength=int(n_clusters))
    zp = PCA(2).fit(z)
    pz, pc = zp.transform(z), zp.transform(km.cluster_centers_)
    order = np.argsort(pc[:, 0])
    dest = out / f'clusters_k{n_clusters}'
    paths = generate_from_latents(km.cluster_centers_[order], dest, [f'cluster_{i}' for i in range(int(n_clusters))], 'clustering')
    labels = np.empty_like(km.labels_)
    for new, old in enumerate(order):
        labels[km.labels_ == old] = new
    np.save(dest / 'cluster_labels.npy', labels)
    sub = Path(STATE['analysis_run']) / 'particles_subset_indices.npy'
    rows = np.load(sub) if sub.exists() else np.arange(len(labels))
    pd.DataFrame({'particle': np.arange(len(labels)), 'row_in_original_metadata': rows, 'cluster': labels}).to_csv(
        dest / 'cluster_per_particle.csv', index=False)
    pd.DataFrame({'cluster': range(int(n_clusters)), 'particles': counts[order], 'pdb': [p.name for p in paths]}).to_csv(dest / 'clusters.csv', index=False)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.3), gridspec_kw={'width_ratios': [1.4, 1]})
    for i in range(int(n_clusters)):
        sel = labels == i
        axes[0].scatter(pz[sel, 0], pz[sel, 1], s=2, alpha=0.3, color=SEG_COLORS[i % len(SEG_COLORS)])
        axes[0].scatter(*pc[order[i]], s=120, color=SEG_COLORS[i % len(SEG_COLORS)], edgecolor='k', zorder=3)
        axes[0].annotate(str(i), pc[order[i]], fontsize=11, weight='bold', xytext=(4, 4), textcoords='offset points')
    axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2'); axes[0].set_title('Clusters in the latent space (centers numbered along PC1)')
    axes[1].bar(range(int(n_clusters)), counts[order], color=[SEG_COLORS[i % len(SEG_COLORS)] for i in range(int(n_clusters))])
    axes[1].set_xlabel('cluster'); axes[1].set_ylabel('particles'); axes[1].set_title('Cluster populations')
    axes[1].set_xticks(range(int(n_clusters)))
    fig.tight_layout()
    plt.show()
    ok(f'{n_clusters} representative structures in {dest} (cluster_<i>.pdb); cluster of every particle in cluster_per_particle.csv '
       '(with its row in your original metadata file)')
    polys = [load_polymer(p) for p in paths]
    base = polys[0]
    disp = np.max([np.linalg.norm(p.coord - base.coord, axis=1) for p in polys], axis=0)
    print('Cycling through the cluster representatives in order; red = residues that differ most between clusters.')
    view3d([p.coord for p in polys], None, base.chain_id, base.res_id, base.res_name, bfactors=disp).show()

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

In [ ]:
#@title 6.6 · Turn structures into density maps (.mrc) { display-mode: "form" }
#@markdown Renders structures from 6.1, 6.4 or 6.5 as density maps with the same Gaussian model cryoSPHERE trains with (one Gaussian per residue, weighted by its electrons).
source = "PC traversal"  #@param ["PC traversal", "Cluster representatives", "Chosen particles"]
principal_component = 1  #@param {type:"integer"}
grid = "Full particle box"  #@param ["Full particle box", "Training grid (Npix_downsize)"]
max_structures = 50  #@param {type:"integer"}

def _main():
    need('analysis_dir')
    out, run_dir = Path(STATE['analysis_dir']), Path(STATE['analysis_run'])
    if source == 'PC traversal':
        folders = [out / f'pc{int(principal_component) - 1}']
    elif source == 'Cluster representatives':
        folders = sorted(out.glob('clusters_k*'), key=lambda p: p.stat().st_mtime)[-1:]
    else:
        folders = sorted(out.glob('particles_*'), key=lambda p: p.stat().st_mtime)[-1:]
    pdbs = [p for f in folders for p in sorted(f.glob('*.pdb'))] if folders else []
    if not pdbs:
        fail(f'No structures found for "{source}"; run the matching step (6.1, 6.5 or 6.4) first.')
    if len(pdbs) > max_structures:
        info(f'Converting the first {max_structures} of {len(pdbs)} structures (raise max_structures for more).')
        pdbs = pdbs[:max_structures]
    images = yaml.safe_load((run_dir / 'images.yaml').read_text())
    if grid.startswith('Full'):
        images['Npix_downsize'] = images['Npix']
    yml = out / '_volume_grid.yaml'
    write_yaml(yml, images)
    (out / '_pdb_list.txt').write_text('\n'.join(str(p) for p in pdbs))
    # Render in a separate process: its GPU memory is released when it exits, and while a background
    # training runs the GPU is left alone (rendering then uses the CPU).
    # Same Gaussian model as training: one Gaussian per residue, weighted by the electrons of the full-atom base residue
    # (cryosphere_structure_to_volume would weight every C-alpha of a predicted PDB equally).
    script = """
import sys, warnings
warnings.filterwarnings('ignore')
import numpy as np, torch, yaml
import cryosphere.data.mrc as mrc
from cryosphere.model import renderer
from cryosphere.model.gmm import EMAN2Grid
from cryosphere.model.polymer import Polymer
images = yaml.safe_load(open(sys.argv[1]))
paths = open(sys.argv[2]).read().splitlines()
base = Polymer.from_pdb(sys.argv[3])
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n = int(images['Npix_downsize'])
apix = images['Npix'] * images['apix'] / n
grid = EMAN2Grid(n, apix, device=dev)
amp = torch.tensor(base.num_electron, dtype=torch.float32, device=dev)[:, None]
sig = torch.ones_like(amp) * images['sigma_gmm']
origin = -n // 2 * apix
for i, p in enumerate(paths):
    xyz = torch.tensor(Polymer.from_pdb(p).coord, dtype=torch.float32, device=dev)[None]
    if xyz.shape[1] != amp.shape[0]:
        sys.exit(f'{p} has {xyz.shape[1]} residues, the base structure {amp.shape[0]}')
    with torch.no_grad():
        vol = renderer.structure_to_volume(xyz, sig, amp, grid, dev)
    mrc.MRCFile.write(p[:-4] + '.mrc', np.transpose(vol[0].cpu().numpy(), (2, 1, 0)), Apix=apix, is_vol=True,
                      xorg=origin, yorg=origin, zorg=origin)
    print(f'{i + 1}/{len(paths)} maps rendered', flush=True)
"""
    base_pdb = yaml.safe_load((run_dir / 'parameters.yaml').read_text())['base_structure_path']
    env = {'CUDA_VISIBLE_DEVICES': ''} if background_run_alive() else {}
    if env:
        info('A background training is running, so the maps are rendered on the CPU (slower).')
    status = display(HTML('<code>Rendering maps…</code>'), display_id=True)
    def show(line):
        status.update(HTML(f'<code>{line}</code>'))
        return False
    rc, tail = stream_process([sys.executable, '-c', script, yml, out / '_pdb_list.txt', base_pdb], out / 'maps_stdout.log',
                              env=env, echo_filter=show, show_cmd=False)
    if rc != 0:
        print('\n'.join(tail)[-3000:])
        fail('Rendering the maps failed (see the error above).')
    apix = images['Npix'] * images['apix'] / images['Npix_downsize']
    ok(f'{len(pdbs)} maps ({images["Npix_downsize"]}³ voxels at {apix:.2f} Å) written next to the PDB files in '
       f'{", ".join(str(f) for f in folders)}')
    info('These maps and all predicted PDBs are in cryoSPHERE\'s frame (box centre at the origin). They overlay the '
         '*_centered.mrc map from 3.1; 7.1 can add copies moved back to your original map frame.')
    info('In ChimeraX: open the .mrc files, then e.g. "volume #1-10 level 0.5 style surface" and "mseries slider #1-10" to flip through them.')

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

---
## 7 · Export

In [ ]:
#@title 7.1 · Download or save your results { display-mode: "form" }
what = "Configs, logs, analysis + latest checkpoint"  #@param ["Configs, logs, analysis + latest checkpoint", "Everything (all kept checkpoints)", "Configs, logs and analysis only"]
destination = "Download to this computer"  #@param ["Download to this computer", "Copy to Google Drive"]
run_to_export = ""  #@param {type:"string"}
#@markdown Add copies of the predicted structures and maps moved back to the frame of your original consensus map (from 3.1's centering):
add_original_frame_copies = True  #@param {type:"boolean"}

def _main():
    run_dir = runs_dir() / run_to_export.strip() if run_to_export.strip() else (Path(STATE['run_dir']) if STATE.get('run_dir') else None)
    if run_dir is None or not run_dir.exists():
        fail('No run to export.')
    eps = list_checkpoints(run_dir)
    keep = set()
    if what.startswith('Everything'):
        keep = {p for pair in eps.values() for p in pair}
    elif what.startswith('Configs, logs, analysis +') and eps:
        keep = set(eps[max(eps)])
    files = {}
    for root in dict.fromkeys([run_dir, run_work_dir(run_dir)]):
        for p in Path(root).rglob('*'):
            if not p.is_file() or p.name.startswith('.') or p.name.endswith('.partial'):
                continue
            if p.name.startswith('ckpt') and p.suffix == '.pt' and p not in keep:
                continue
            files.setdefault(str(p.relative_to(root)), p)
    tmp = LOCAL_ROOT
    zpath = tmp / f'{run_dir.name}_results.zip'
    skipped = 0
    shift = None
    if add_original_frame_copies and (run_dir / 'centering.json').exists():
        shift = np.array(json.loads((run_dir / 'centering.json').read_text())['shift_A'], dtype=float)
        if not np.any(shift):
            shift = None
    moved = 0
    with zipfile.ZipFile(zpath, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        for rel, p in sorted(files.items()):
            try:
                zf.write(p, Path(run_dir.name) / rel)
                if shift is not None and rel.startswith('analysis') and p.suffix in ('.pdb', '.mrc') and moved < 2000 \
                        and '_config' not in rel:
                    tmp_copy = LOCAL_ROOT / '_original_frame_tmp' / p.name
                    tmp_copy.parent.mkdir(exist_ok=True)
                    to_original_frame(p, tmp_copy, shift)
                    zf.write(tmp_copy, Path(run_dir.name) / 'original_frame' / rel)
                    tmp_copy.unlink()
                    moved += 1
            except FileNotFoundError:          # pruned by a running training in the meantime
                skipped += 1
    if moved:
        info(f'{moved} structures/maps also saved under original_frame/ (shifted by {-shift.round(2)} Å to your map\'s frame).')
    files = [f for f in files if f]
    if skipped:
        info(f'{skipped} checkpoint(s) were removed by the running training while packing and are not included.')
    ok(f'{len(files)} files packed into {zpath.name} ({human_size(zpath.stat().st_size)})')
    if destination.startswith('Download'):
        if not IN_COLAB:
            info(f'Not in Colab: the archive is at {zpath}')
            return
        from google.colab import files as colab_files
        if zpath.stat().st_size > 2e9:
            warn('Browser downloads above ~2 GB often fail; consider "Copy to Google Drive".')
        colab_files.download(str(zpath))
    else:
        if not IN_COLAB:
            fail('Google Drive is only available inside Colab.')
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            drive.mount('/content/drive')
        dest = Path('/content/drive/MyDrive/cryoSPHERE/exports')
        dest.mkdir(parents=True, exist_ok=True)
        shutil.copy(zpath, dest / zpath.name)
        ok(f'Copied to {dest / zpath.name}')

if 'CSH_LOADED' not in globals():
    raise RuntimeError('Run step 1.2 (Load the notebook helpers) first.')
run_step(_main)

---
## 8 · Troubleshooting

| Symptom | Cause and fix |
|---|---|
| `No CUDA GPU is available` | **Runtime → Change runtime type → T4 GPU**. cryoSPHERE cannot run on CPU. |
| `CUDA out of memory` | Lower `batch_size` (e.g. 64), downsample (`Npix_downsize` 128), or use the *Light* clashing loss in 4.1. |
| `Address already in use` / training hangs at start | cryoSPHERE always uses port 12355, so only one training or analysis can run at a time. Stop the other job in 5.3; if the port stays busy, use **Runtime → Disconnect and delete runtime** and start again from 1.1 (a Drive project keeps your runs). |
| Training stopped: loss became `nan` | Almost always too many segments for the number of residues: use at least ~50 residues per segment (`N_segments = 0` chooses this) and train a new run. Only if that does not help, check the defocus/CTF columns of your metadata. |
| `No such file or directory` after reconnecting | A new runtime starts with an empty Colab disk and without cryoSPHERE installed. Run 1.1–1.4 (1.4 lists exactly what is missing), then 2.1, 2.2 and 3.1 with the same settings. With project storage on Drive your runs and checkpoints survive. |
| W&B `API key must be 40 characters` | New W&B keys are longer; 1.5 upgrades `wandb` automatically. Or switch W&B off. |
| Structures look exploded or do not move | Check 3.2: the model must be centered and fit the map. Try fewer segments, a stronger low-pass filter (`lp_bandwidth`), or more epochs. |
| Colab disconnected during training | Re-open the notebook and run 1.1–1.4 (same Drive project), then 2.1, 2.2 and 3.1 with the same settings, then 4.1 with *Continue a previous run* and a new `run_name`, then 5.1. |

**Running cryoSPHERE outside Colab.** Every step prints the exact command it runs. On your own GPU machine the same workflow is:
```bash
pip install cryosphere
cryosphere_center_origin --pdb_file_path fitted.pdb --mrc_file_path consensus_map.mrc
cryosphere_structure_to_volume --image_yaml images.yaml --structure_path fitted_centered.pdb --output_path fitted_gmm.mrc
cryosphere_train --experiment_yaml parameters.yaml
# analysis: use a copy of parameters.yaml whose folder_path is another directory, otherwise it overwrites the training run.log
cryosphere_analyze --experiment_yaml analysis_parameters.yaml --model cryoSPHERE/ckpt49.pt --segmenter cryoSPHERE/seg49.pt --output_path analysis --no-generate_structures
```
The run folder exported in 7.1 contains `parameters.yaml`, `images.yaml`, the centred base structure and `centering.json`; edit the paths in `parameters.yaml` to use them there.

**Citation.** If cryoSPHERE helps your research, please cite: Ducrocq *et al.*, *cryoSPHERE: Single-particle heterogeneous reconstruction from cryo EM*, ICLR 2025 ([arXiv:2407.01574](https://arxiv.org/abs/2407.01574)).